# Retail Sales Performance Analysis — Data Profiling

## Objective

The purpose of this notebook is to inspect and validate the raw Online Retail II dataset before performing any data cleaning or transformation.

The initial analysis focuses on:

- Dataset structure and dimensions
- Column names and data types
- Missing values
- Duplicate records
- Invalid or unusual quantities and prices
- Transaction date coverage
- Potential cancellations, returns, and non-product transactions

In [3]:
%pip install pandas openpyxl

75.62s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd

In [3]:
from pathlib import Path

file_path = Path("../data/raw/online_retail_II.xlsx")

print("File exists:", file_path.exists())
print("File size:", round(file_path.stat().st_size / (1024 ** 2), 2), "MB")

File exists: True
File size: 43.51 MB


In [4]:
sample_2009_2010 = pd.read_excel(
    file_path,
    sheet_name="Year 2009-2010",
    nrows=5
)

sample_2009_2010

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom


In [5]:
print(sample_2009_2010.columns.tolist())
print(sample_2009_2010.dtypes)

['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']
Invoice                 int64
StockCode              object
Description               str
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID             int64
Country                   str
dtype: object


## Full Dataset Ingestion

The complete worksheets are loaded with identifier fields treated as strings to preserve their categorical meaning and avoid unintended numerical interpretation.

In [6]:
import time

start = time.perf_counter()

sales_2009_2010 = pd.read_excel(
    file_path,
    sheet_name="Year 2009-2010",
    dtype={
        "Invoice": "string",
        "StockCode": "string",
        "Description": "string",
        "Customer ID": "string",
        "Country": "string"
    }
)

elapsed = time.perf_counter() - start

print("Shape:", sales_2009_2010.shape)
print(f"Load time: {elapsed:.1f} seconds")

Shape: (525461, 8)
Load time: 11.9 seconds


In [7]:
sales_2009_2010.dtypes

Invoice                string
StockCode              string
Description            string
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID            string
Country                string
dtype: object

In [8]:
start = time.perf_counter()

sales_2010_2011 = pd.read_excel(
    file_path,
    sheet_name="Year 2010-2011",
    dtype={
        "Invoice": "string",
        "StockCode": "string",
        "Description": "string",
        "Customer ID": "string",
        "Country": "string"
    }
)

elapsed = time.perf_counter() - start

print("Shape:", sales_2010_2011.shape)
print(f"Load time: {elapsed:.1f} seconds")

Shape: (541910, 8)
Load time: 12.7 seconds


## Combine Annual Worksheets

The two annual worksheets are combined into a single transactional dataset. A source-period field is retained so records can be traced back to their original worksheet during validation.

In [9]:
sales_2009_2010["SourcePeriod"] = "2009-2010"
sales_2010_2011["SourcePeriod"] = "2010-2011"

sales = pd.concat(
    [sales_2009_2010, sales_2010_2011],
    ignore_index=True
)

print("Combined shape:", sales.shape)

Combined shape: (1067371, 9)


In [11]:
sales.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourcePeriod
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,2009-2010
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,2009-2010
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,2009-2010
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,2009-2010
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,2009-2010


In [12]:
sales.tail()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourcePeriod
1067366,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680,France,2010-2011
1067367,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680,France,2010-2011
1067368,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680,France,2010-2011
1067369,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680,France,2010-2011
1067370,581587,POST,POSTAGE,1,2011-12-09 12:50:00,18.00,12680,France,2010-2011


In [13]:
sales.dtypes

Invoice                 string
StockCode               string
Description             string
Quantity                 int64
InvoiceDate     datetime64[us]
Price                  float64
Customer ID             string
Country                 string
SourcePeriod               str
dtype: object

In [14]:
duplicate_rows = sales.duplicated(
    subset=[
        "Invoice",
        "StockCode",
        "Description",
        "Quantity",
        "InvoiceDate",
        "Price",
        "Customer ID",
        "Country"
    ]
).sum()

print("Exact duplicate transaction rows:", duplicate_rows)

Exact duplicate transaction rows: 34335


## Duplicate Record Investigation

Exact duplicate transaction records were identified in the combined dataset. Before removing any records, duplicates are evaluated separately within each annual worksheet and across worksheets to determine whether they represent source overlap or duplicated transaction entries.

In [15]:
key_columns = [
    "Invoice",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "Price",
    "Customer ID",
    "Country"
]

duplicates_2009_2010 = sales_2009_2010.duplicated(
    subset=key_columns
).sum()

duplicates_2010_2011 = sales_2010_2011.duplicated(
    subset=key_columns
).sum()

print("Duplicates within 2009-2010:", duplicates_2009_2010)
print("Duplicates within 2010-2011:", duplicates_2010_2011)

Duplicates within 2009-2010: 6865
Duplicates within 2010-2011: 5268


In [16]:
unique_2009_2010 = sales_2009_2010[key_columns].drop_duplicates()
unique_2010_2011 = sales_2010_2011[key_columns].drop_duplicates()

cross_sheet_matches = unique_2009_2010.merge(
    unique_2010_2011,
    on=key_columns,
    how="inner"
)

print(
    "Unique transaction-line patterns appearing in both sheets:",
    len(cross_sheet_matches)
)

Unique transaction-line patterns appearing in both sheets: 22202


In [17]:
print(
    "Earliest cross-sheet match:",
    cross_sheet_matches["InvoiceDate"].min()
)

print(
    "Latest cross-sheet match:",
    cross_sheet_matches["InvoiceDate"].max()
)

Earliest cross-sheet match: 2010-12-01 08:26:00
Latest cross-sheet match: 2010-12-09 20:01:00


In [18]:
duplicate_examples = sales[
    sales.duplicated(
        subset=key_columns,
        keep=False
    )
].sort_values(
    ["InvoiceDate", "Invoice", "StockCode"]
)

duplicate_examples[
    [
        "Invoice",
        "StockCode",
        "Description",
        "Quantity",
        "InvoiceDate",
        "Price",
        "Customer ID",
        "Country",
        "SourcePeriod"
    ]
].head(20)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourcePeriod
379,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329,United Kingdom,2009-2010
391,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329,United Kingdom,2009-2010
365,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329,United Kingdom,2009-2010
386,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329,United Kingdom,2009-2010
363,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329,United Kingdom,2009-2010
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329,United Kingdom,2009-2010
394,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329,United Kingdom,2009-2010
362,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329,United Kingdom,2009-2010
385,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329,United Kingdom,2009-2010
368,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01 11:34:00,0.85,16329,United Kingdom,2009-2010


## Cross-Sheet Overlap Validation

The annual worksheets overlap between December 1 and December 9, 2010. The overlapping period is compared directly to determine whether the second worksheet is a duplicate continuation of the first or whether either worksheet contains unique transaction records.

In [19]:
overlap_start = pd.Timestamp("2010-12-01")
overlap_end = pd.Timestamp("2010-12-09 23:59:59")

overlap_2009_2010 = sales_2009_2010[
    sales_2009_2010["InvoiceDate"].between(
        overlap_start,
        overlap_end
    )
]

overlap_2010_2011 = sales_2010_2011[
    sales_2010_2011["InvoiceDate"].between(
        overlap_start,
        overlap_end
    )
]

print("2009-2010 overlap rows:", overlap_2009_2010.shape)
print("2010-2011 overlap rows:", overlap_2010_2011.shape)

2009-2010 overlap rows: (22523, 9)
2010-2011 overlap rows: (22523, 9)


In [20]:
overlap_first = (
    overlap_2009_2010[key_columns]
    .drop_duplicates()
)

overlap_second = (
    overlap_2010_2011[key_columns]
    .drop_duplicates()
)

print("Unique rows in first overlap:", len(overlap_first))
print("Unique rows in second overlap:", len(overlap_second))

Unique rows in first overlap: 22202
Unique rows in second overlap: 22202


In [21]:
only_in_first = overlap_first.merge(
    overlap_second,
    on=key_columns,
    how="left",
    indicator=True
)

only_in_first = only_in_first[
    only_in_first["_merge"] == "left_only"
]

only_in_second = overlap_second.merge(
    overlap_first,
    on=key_columns,
    how="left",
    indicator=True
)

only_in_second = only_in_second[
    only_in_second["_merge"] == "left_only"
]

print("Unique transaction patterns only in 2009-2010 sheet:", len(only_in_first))
print("Unique transaction patterns only in 2010-2011 sheet:", len(only_in_second))

Unique transaction patterns only in 2009-2010 sheet: 0
Unique transaction patterns only in 2010-2011 sheet: 0


### Overlap Resolution Decision

The two source worksheets contain an identical set of 22,202 unique transaction-line patterns between December 1 and December 9, 2010. To prevent double-counting, records from the 2009-2010 worksheet are retained only through November 30, 2010, while the 2010-2011 worksheet is used from December 1, 2010 onward.

In [22]:
cutoff_date = pd.Timestamp("2010-12-01")

sales_2009_2010_trimmed = sales_2009_2010[
    sales_2009_2010["InvoiceDate"] < cutoff_date
].copy()

sales_base = pd.concat(
    [sales_2009_2010_trimmed, sales_2010_2011],
    ignore_index=True
)

print("Original combined rows:", len(sales))
print("Rows after resolving overlap:", len(sales_base))
print("Rows removed due to source overlap:", len(sales) - len(sales_base))

Original combined rows: 1067371
Rows after resolving overlap: 1044848
Rows removed due to source overlap: 22523


In [23]:
duplicates_after_overlap = sales_base.duplicated(
    subset=key_columns
).sum()

print(
    "Exact duplicate rows after resolving worksheet overlap:",
    duplicates_after_overlap
)

Exact duplicate rows after resolving worksheet overlap: 11812


In [24]:
duplicate_records = sales_base[
    sales_base.duplicated(
        subset=key_columns,
        keep=False
    )
].copy()

duplicate_records["LineValue"] = (
    duplicate_records["Quantity"]
    * duplicate_records["Price"]
)

print("Rows involved in duplicate groups:", len(duplicate_records))
print(
    "Potential line value represented by duplicate-group rows:",
    round(duplicate_records["LineValue"].sum(), 2)
)

Rows involved in duplicate groups: 22813
Potential line value represented by duplicate-group rows: 105183.71


In [25]:
extra_duplicates = sales_base[
    sales_base.duplicated(
        subset=key_columns,
        keep="first"
    )
].copy()

extra_duplicates["LineValue"] = (
    extra_duplicates["Quantity"]
    * extra_duplicates["Price"]
)

print("Extra duplicate rows:", len(extra_duplicates))
print(
    "Potential duplicated line value:",
    round(extra_duplicates["LineValue"].sum(), 2)
)

Extra duplicate rows: 11812
Potential duplicated line value: 54228.42


### Exact Duplicate Resolution

After resolving the worksheet overlap, exact duplicate transaction-line records remained within the retained source data. Because these records contain identical values across all available transaction fields and no unique line-item identifier exists to distinguish them, duplicate occurrences are removed while retaining the first recorded instance. This prevents duplicate quantities and transaction values from being counted multiple times in downstream analysis.

In [26]:
sales_dedup = sales_base.drop_duplicates(
    subset=key_columns,
    keep="first"
).copy()

print("Rows before duplicate removal:", len(sales_base))
print("Duplicate occurrences removed:", len(sales_base) - len(sales_dedup))
print("Rows after duplicate removal:", len(sales_dedup))

Rows before duplicate removal: 1044848
Duplicate occurrences removed: 11812
Rows after duplicate removal: 1033036


In [27]:
remaining_duplicates = sales_dedup.duplicated(
    subset=key_columns
).sum()

print("Remaining exact duplicates:", remaining_duplicates)

Remaining exact duplicates: 0


## Missing Value Analysis

Missing values are evaluated by column to determine their frequency, percentage, and potential impact on sales, customer, and product-level analysis. Missing records are not removed automatically; treatment decisions are based on how each field is used in downstream analysis.

In [28]:
missing_summary = pd.DataFrame({
    "Missing_Count": sales_dedup.isna().sum(),
    "Missing_Percentage": (
        sales_dedup.isna().mean() * 100
    ).round(2)
})

missing_summary

,Missing_Count,Missing_Percentage
Invoice,0,0.00
StockCode,0,0.00
Description,4275,0.41
Quantity,0,0.00
InvoiceDate,0,0.00
Price,0,0.00
Customer ID,235151,22.76
Country,0,0.00
SourcePeriod,0,0.00


In [29]:
print(
    "Missing Customer IDs:",
    sales_dedup["Customer ID"].isna().sum()
)

print(
    "Missing Descriptions:",
    sales_dedup["Description"].isna().sum()
)

Missing Customer IDs: 235151
Missing Descriptions: 4275


In [30]:
sales_dedup[
    sales_dedup["Customer ID"].isna()
].head(20)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourcePeriod
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.00,<NA>,United Kingdom,2009-2010
283,489463,71477,short,-240,2009-12-01 10:52:00,0.00,<NA>,United Kingdom,2009-2010
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.00,<NA>,United Kingdom,2009-2010
470,489521,21646,<NA>,-50,2009-12-01 11:44:00,0.00,<NA>,United Kingdom,2009-2010
577,489525,85226C,BLUE PULL BACK RACING CAR,1,2009-12-01 11:49:00,0.55,<NA>,United Kingdom,2009-2010
578,489525,85227,SET/6 3D KIT CARDS FOR KIDS,1,2009-12-01 11:49:00,0.85,<NA>,United Kingdom,2009-2010
1055,489548,22271,FELTCRAFT DOLL ROSIE,1,2009-12-01 12:32:00,2.95,<NA>,United Kingdom,2009-2010
1056,489548,22254,FELT TOADSTOOL LARGE,12,2009-12-01 12:32:00,1.25,<NA>,United Kingdom,2009-2010
1057,489548,22273,FELTCRAFT DOLL MOLLY,3,2009-12-01 12:32:00,2.95,<NA>,United Kingdom,2009-2010
1058,489548,22195,LARGE HEART MEASURING SPOONS,1,2009-12-01 12:32:00,1.65,<NA>,United Kingdom,2009-2010


In [31]:
sales_dedup[
    sales_dedup["Description"].isna()
].head(20)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourcePeriod
470,489521,21646,<NA>,-50,2009-12-01 11:44:00,0.0,<NA>,United Kingdom,2009-2010
3114,489655,20683,<NA>,-44,2009-12-01 17:26:00,0.0,<NA>,United Kingdom,2009-2010
3161,489659,21350,<NA>,230,2009-12-01 17:39:00,0.0,<NA>,United Kingdom,2009-2010
3731,489781,84292,<NA>,17,2009-12-02 11:45:00,0.0,<NA>,United Kingdom,2009-2010
4296,489806,18010,<NA>,-770,2009-12-02 12:42:00,0.0,<NA>,United Kingdom,2009-2010
4566,489821,85049G,<NA>,-240,2009-12-02 13:25:00,0.0,<NA>,United Kingdom,2009-2010
6378,489882,35751C,<NA>,12,2009-12-02 16:22:00,0.0,<NA>,United Kingdom,2009-2010
6555,489898,79323G,<NA>,954,2009-12-03 09:40:00,0.0,<NA>,United Kingdom,2009-2010
6576,489901,21098,<NA>,-200,2009-12-03 09:47:00,0.0,<NA>,United Kingdom,2009-2010
6581,489903,21166,<NA>,48,2009-12-03 09:57:00,0.0,<NA>,United Kingdom,2009-2010


### Missing Description Investigation

Records with missing product descriptions are investigated further to determine whether they represent valid sales transactions or non-sale operational adjustments. Price, customer identification, and quantity behavior are examined before establishing a cleaning rule.

In [32]:
missing_description = sales_dedup[
    sales_dedup["Description"].isna()
].copy()

print("Total missing descriptions:", len(missing_description))

print(
    "Missing description rows with Price = 0:",
    (missing_description["Price"] == 0).sum()
)

print(
    "Missing description rows with Customer ID missing:",
    missing_description["Customer ID"].isna().sum()
)

print(
    "Missing description rows with Quantity < 0:",
    (missing_description["Quantity"] < 0).sum()
)

print(
    "Missing description rows with Quantity = 0:",
    (missing_description["Quantity"] == 0).sum()
)

print(
    "Missing description rows with Quantity > 0:",
    (missing_description["Quantity"] > 0).sum()
)

Total missing descriptions: 4275
Missing description rows with Price = 0: 4275
Missing description rows with Customer ID missing: 4275
Missing description rows with Quantity < 0: 2633
Missing description rows with Quantity = 0: 0
Missing description rows with Quantity > 0: 1642


In [33]:
missing_description[
    missing_description["Price"] > 0
].head(20)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourcePeriod


In [34]:
missing_customer = sales_dedup[
    sales_dedup["Customer ID"].isna()
].copy()

missing_customer["LineValue"] = (
    missing_customer["Quantity"]
    * missing_customer["Price"]
)

print("Rows with missing Customer ID:", len(missing_customer))

print(
    "Rows with missing Customer ID and positive Price:",
    (missing_customer["Price"] > 0).sum()
)

print(
    "Net line value from missing-Customer-ID records:",
    round(missing_customer["LineValue"].sum(), 2)
)

Rows with missing Customer ID: 235151
Rows with missing Customer ID and positive Price: 229202
Net line value from missing-Customer-ID records: 2565542.41


### Missing Value Treatment Decisions

**Missing Product Description:**  
4,275 records have no product description. All of these records also have a zero price and missing Customer ID, with both positive and negative quantities. These records appear consistent with non-revenue inventory or operational adjustments rather than normal customer sales. They are excluded from the analytical sales dataset to prevent distortion of product and quantity-based metrics.

**Missing Customer ID:**  
235,151 records have no Customer ID. However, 229,202 of these records contain positive prices and collectively represent substantial transaction value. These records are retained for sales, product, geographic, and time-based analysis, but excluded from customer-specific metrics such as unique customers, repeat customers, and customer segmentation.

In [35]:
sales_working = sales_dedup[
    sales_dedup["Description"].notna()
].copy()

sales_working["CustomerKnown"] = (
    sales_working["Customer ID"].notna()
)

print("Rows before missing-description removal:", len(sales_dedup))
print(
    "Missing-description rows removed:",
    len(sales_dedup) - len(sales_working)
)
print("Working dataset rows:", len(sales_working))

Rows before missing-description removal: 1033036
Missing-description rows removed: 4275
Working dataset rows: 1028761


In [36]:
print(
    "Remaining missing descriptions:",
    sales_working["Description"].isna().sum()
)

print(
    "Missing Customer IDs retained:",
    sales_working["Customer ID"].isna().sum()
)

Remaining missing descriptions: 0
Missing Customer IDs retained: 230876


## Returns and Cancellation Investigation

Negative quantities and cancellation-style invoice numbers are evaluated to determine how returned or cancelled transactions should be classified and incorporated into revenue and return-rate metrics.

In [37]:
sales_working["IsCancellationInvoice"] = (
    sales_working["Invoice"]
    .str.startswith("C", na=False)
)

print(
    "Rows with negative Quantity:",
    (sales_working["Quantity"] < 0).sum()
)

print(
    "Rows with zero Quantity:",
    (sales_working["Quantity"] == 0).sum()
)

print(
    "Rows with positive Quantity:",
    (sales_working["Quantity"] > 0).sum()
)

print(
    "Rows with cancellation invoice:",
    sales_working["IsCancellationInvoice"].sum()
)

Rows with negative Quantity: 19863
Rows with zero Quantity: 0
Rows with positive Quantity: 1008898
Rows with cancellation invoice: 19104


In [38]:
cancellation_check = pd.crosstab(
    sales_working["IsCancellationInvoice"],
    sales_working["Quantity"] < 0,
    rownames=["Cancellation Invoice"],
    colnames=["Negative Quantity"]
)

cancellation_check

Negative Quantity,False,True
Cancellation Invoice,,
False,1008897,760
True,1,19103


In [39]:
sales_working[
    sales_working["IsCancellationInvoice"]
][
    [
        "Invoice",
        "StockCode",
        "Description",
        "Quantity",
        "InvoiceDate",
        "Price",
        "Customer ID",
        "Country"
    ]
].head(20)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321,Australia
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321,Australia
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321,Australia
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321,Australia
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321,Australia
183,C489449,21871,SAVE THE PLANET MUG,-12,2009-12-01 10:33:00,1.25,16321,Australia
184,C489449,84946,ANTIQUE SILVER TEA GLASS ETCHED,-12,2009-12-01 10:33:00,1.25,16321,Australia
185,C489449,84970S,HANGING HEART ZINC T-LIGHT HOLDER,-24,2009-12-01 10:33:00,0.85,16321,Australia
186,C489449,22090,PAPER BUNTING RETRO SPOTS,-12,2009-12-01 10:33:00,2.95,16321,Australia
196,C489459,90200A,PURPLE SWEETHEART BRACELET,-3,2009-12-01 10:44:00,4.25,17592,United Kingdom


### Negative Quantity Exceptions

Negative quantities that do not belong to cancellation-style invoices are investigated separately to determine whether they represent returns, inventory adjustments, corrections, or other non-sale activity.

In [40]:
negative_non_cancellation = sales_working[
    (sales_working["Quantity"] < 0)
    & (~sales_working["IsCancellationInvoice"])
].copy()

print(
    "Negative quantity rows without C invoice:",
    len(negative_non_cancellation)
)

negative_non_cancellation[
    [
        "Invoice",
        "StockCode",
        "Description",
        "Quantity",
        "InvoiceDate",
        "Price",
        "Customer ID",
        "Country"
    ]
].head(30)

Negative quantity rows without C invoice: 760


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.0,<NA>,United Kingdom
283,489463,71477,short,-240,2009-12-01 10:52:00,0.0,<NA>,United Kingdom
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.0,<NA>,United Kingdom
3162,489660,35956,lost,-1043,2009-12-01 17:43:00,0.0,<NA>,United Kingdom
3168,489663,35605A,damages,-117,2009-12-01 18:02:00,0.0,<NA>,United Kingdom
4538,489820,21133,invcd as 84879?,-720,2009-12-02 13:23:00,0.0,<NA>,United Kingdom
6556,489899,79323GR,sold as gold,-954,2009-12-03 09:41:00,0.0,<NA>,United Kingdom
6911,490007,84347,21494,-720,2009-12-03 12:09:00,0.0,<NA>,United Kingdom
9308,490130,21493,lost?,-600,2009-12-03 18:28:00,0.0,<NA>,United Kingdom
17427,490765,21450,damaged,-31,2009-12-08 10:59:00,0.0,<NA>,United Kingdom


In [41]:
print(
    "Price = 0:",
    (negative_non_cancellation["Price"] == 0).sum()
)

print(
    "Price > 0:",
    (negative_non_cancellation["Price"] > 0).sum()
)

print(
    "Missing Customer ID:",
    negative_non_cancellation["Customer ID"].isna().sum()
)

Price = 0: 760
Price > 0: 0
Missing Customer ID: 760


In [42]:
negative_non_cancellation[
    "Description"
].value_counts().head(20)

Description
check                     121
damages                    83
?                          81
damaged                    78
missing                    27
sold as set on dotcom      20
Damaged                    17
smashed                     9
thrown away                 9
Unsaleable, destroyed.      9
dotcom                      8
damages?                    7
??                          7
crushed                     6
given away                  6
MIA                         5
Damages                     5
counted                     5
checked                     5
wet damaged                 5
Name: count, dtype: Int64

In [43]:
negative_non_cancellation[
    "StockCode"
].value_counts().head(20)

StockCode
22423     10
84016      5
20852      5
85175      5
47566B     4
22719      4
21843      4
85172      4
21830      4
72802C     4
21100      3
82494L     3
21463      3
79000      3
21735      3
20713      3
37479P     3
82582      3
84406B     3
46000S     3
Name: count, dtype: Int64

In [44]:
sales_working[
    sales_working["IsCancellationInvoice"]
    & (sales_working["Quantity"] >= 0)
][
    [
        "Invoice",
        "StockCode",
        "Description",
        "Quantity",
        "InvoiceDate",
        "Price",
        "Customer ID",
        "Country"
    ]
]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
76799,C496350,M,Manual,1,2010-02-01 08:24:00,373.57,<NA>,United Kingdom


### Returns and Adjustment Classification

The investigation identified distinct transaction behaviors:

- Standard cancellation invoices generally begin with `C` and contain negative quantities.
- 760 negative-quantity records do not use cancellation-style invoice numbers. All have zero prices and missing Customer IDs, with descriptions such as damaged, missing, smashed, thrown away, or other stock-related notes. These records are treated as operational inventory adjustments rather than customer returns.
- One cancellation-style invoice contains a positive quantity and is recorded as a manual transaction. This record is treated as an exceptional/manual adjustment rather than a standard sale or return.

This classification prevents operational stock movements from being incorrectly included in sales or customer-return metrics.

In [46]:
sales_working["TransactionType"] = "Sale"

sales_working.loc[
    sales_working["IsCancellationInvoice"]
    & (sales_working["Quantity"] < 0),
    "TransactionType"
] = "Cancellation/Return"

sales_working.loc[
    (~sales_working["IsCancellationInvoice"])
    & (sales_working["Quantity"] < 0)
    & (sales_working["Price"] == 0),
    "TransactionType"
] = "Operational Adjustment"

sales_working.loc[
    sales_working["IsCancellationInvoice"]
    & (sales_working["Quantity"] >= 0),
    "TransactionType"
] = "Manual/Exceptional Adjustment"

In [47]:
sales_working["TransactionType"].value_counts()

TransactionType
Sale                             1008897
Cancellation/Return                19103
Operational Adjustment               760
Manual/Exceptional Adjustment          1
Name: count, dtype: int64

In [48]:
pd.crosstab(
    sales_working["TransactionType"],
    [
        sales_working["Quantity"] < 0,
        sales_working["Price"] == 0
    ]
)

Quantity                         False        True       
Price                            False True   False True 
TransactionType                                          
Cancellation/Return                  0     0  19103     0
Manual/Exceptional Adjustment        1     0      0     0
Operational Adjustment               0     0      0   760
Sale                           1007918   979      0     0

## Zero Price Investigation

Positive-quantity transactions with a zero unit price are investigated to determine whether they represent promotional items, free samples, operational records, or invalid sales transactions. These records are reviewed before defining the final revenue-generating sales dataset.

In [49]:
zero_price_sales = sales_working[
    (sales_working["TransactionType"] == "Sale")
    & (sales_working["Price"] == 0)
].copy()

print("Zero-price sale rows:", len(zero_price_sales))

zero_price_sales[
    [
        "Invoice",
        "StockCode",
        "Description",
        "Quantity",
        "InvoiceDate",
        "Price",
        "Customer ID",
        "Country"
    ]
].head(30)

Zero-price sale rows: 979


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
4674,489825,22076,6 RIBBONS EMPIRE,12,2009-12-02 13:34:00,0.0,16126,United Kingdom
5904,489861,DOT,DOTCOM POSTAGE,1,2009-12-02 14:50:00,0.0,<NA>,United Kingdom
6781,489998,48185,DOOR MAT FAIRY CAKE,2,2009-12-03 11:19:00,0.0,15658,United Kingdom
16107,490727,M,Manual,1,2009-12-07 16:38:00,0.0,17231,United Kingdom
18738,490961,22065,CHRISTMAS PUDDING TRINKET POT,1,2009-12-08 15:25:00,0.0,14108,United Kingdom
18739,490961,22142,CHRISTMAS CRAFT WHITE FAIRY,12,2009-12-08 15:25:00,0.0,14108,United Kingdom
31993,491971,85042,ANTIQUE LILY FAIRY LIGHTS,1,2009-12-14 18:37:00,0.0,<NA>,United Kingdom
32916,492079,85042,ANTIQUE LILY FAIRY LIGHTS,8,2009-12-15 13:49:00,0.0,15070,United Kingdom
40101,492760,21143,ANTIQUE GLASS HEART DECORATION,12,2009-12-18 14:22:00,0.0,18071,United Kingdom
47126,493761,79320,FLAMINGO LIGHTS,24,2010-01-06 14:54:00,0.0,14258,United Kingdom


In [50]:
print(
    "Zero-price rows with Customer ID:",
    zero_price_sales["Customer ID"].notna().sum()
)

print(
    "Zero-price rows without Customer ID:",
    zero_price_sales["Customer ID"].isna().sum()
)

Zero-price rows with Customer ID: 70
Zero-price rows without Customer ID: 909


In [51]:
zero_price_sales[
    "Description"
].value_counts().head(25)

Description
check                                  39
found                                  28
OWL DOORSTOP                           14
adjustment                             14
POLYESTER FILLER PAD 45x45cm           12
POLYESTER FILLER PAD 40x40cm           10
?                                       9
Found                                   9
FRENCH BLUE METAL DOOR SIGN 1           9
PICNIC BASKET WICKER LARGE              8
BOX OF 24 COCKTAIL PARASOLS             8
RED KITCHEN SCALES                      8
IVORY KITCHEN SCALES                    8
RECIPE BOX PANTRY YELLOW DESIGN         8
amazon                                  8
FRENCH BLUE METAL DOOR SIGN 8           8
Manual                                  7
WATERING CAN BLUE ELEPHANT              7
CHILDS GARDEN SPADE BLUE                7
ENAMEL WASH BOWL CREAM                  7
AIRLINE BAG VINTAGE WORLD CHAMPION      7
MINT KITCHEN SCALES                     7
RED RETROSPOT CHARLOTTE BAG             7
FRENCH BLUE METAL DOOR

In [52]:
zero_price_sales[
    "StockCode"
].value_counts().head(25)

StockCode
21116     14
46000M    12
46000S    10
22501     10
22676      9
20724      8
84692      8
22625      8
22624      8
22666      8
22683      8
M          7
22355      7
22431      7
22366      7
22514      7
22426      7
22372      7
22627      7
22734      7
22679      7
22686      7
22678      7
21523      6
22359      6
Name: count, dtype: Int64

In [53]:
print(
    "Minimum Quantity:",
    zero_price_sales["Quantity"].min()
)

print(
    "Maximum Quantity:",
    zero_price_sales["Quantity"].max()
)

print(
    "Total zero-price units:",
    zero_price_sales["Quantity"].sum()
)

Minimum Quantity: 1
Maximum Quantity: 12540
Total zero-price units: 69421


### Zero-Price Treatment Decision

979 positive-quantity records have a unit price of zero. These records contain a mixture of normal product descriptions, test records, manual entries, stock checks, adjustments, and other operational descriptions. Because the source data does not provide enough information to reliably distinguish promotional items from operational activity, these transactions are classified as non-revenue transactions rather than standard sales.

They are retained for traceability but excluded from revenue-generating sales and units-sold KPIs.

In [54]:
sales_working.loc[
    (sales_working["TransactionType"] == "Sale")
    & (sales_working["Price"] == 0),
    "TransactionType"
] = "Zero-Price/Non-Revenue"

In [55]:
sales_working["TransactionType"].value_counts()

TransactionType
Sale                             1007918
Cancellation/Return                19103
Zero-Price/Non-Revenue               979
Operational Adjustment               760
Manual/Exceptional Adjustment          1
Name: count, dtype: int64

## Non-Product and Special StockCode Investigation

Special stock codes and administrative transaction lines are reviewed separately to distinguish merchandise sales from postage, discounts, manual charges, fees, test records, and other non-product activity.

In [56]:
special_codes = sales_working[
    sales_working["StockCode"].str.fullmatch(
        r"[A-Za-z]+",
        na=False
    )
]

special_codes[
    [
        "StockCode",
        "Description",
        "Quantity",
        "Price",
        "TransactionType"
    ]
].value_counts().head(30)

StockCode  Description                 Quantity  Price   TransactionType    
POST       POSTAGE                      2        18.000  Sale                   329
                                        1        18.000  Sale                   261
                                        3        18.000  Sale                   218
                                        4        18.000  Sale                   108
                                        1        15.000  Sale                    94
                                        5        18.000  Sale                    70
                                        1        40.000  Sale                    69
                                        2        40.000  Sale                    58
                                                 15.000  Sale                    56
                                        6        18.000  Sale                    46
                                        2        28.000  Sale                    39

### Special StockCode Summary

Potential non-merchandise StockCodes are summarized by code and description to distinguish physical products from postage, manual charges, discounts, fees, test records, and other administrative transaction lines.

In [57]:
sales_working["LineValue"] = (
    sales_working["Quantity"] * sales_working["Price"]
)

In [59]:
special_codes = sales_working[
    sales_working["StockCode"].str.fullmatch(
        r"[A-Za-z]+",
        na=False
    )
].copy()

In [60]:
special_code_summary = (
    special_codes
    .groupby(["StockCode", "Description"], dropna=False)
    .agg(
        Rows=("Invoice", "size"),
        TotalQuantity=("Quantity", "sum"),
        NetLineValue=("LineValue", "sum")
    )
    .reset_index()
    .sort_values("Rows", ascending=False)
)

special_code_summary.head(50)

,StockCode,Description,Rows,TotalQuantity,NetLineValue
14,POST,POSTAGE,2079,5061,110430.410
11,DOT,DOTCOM POSTAGE,1423,1417,309844.100
12,M,Manual,1387,4193,-83326.330
6,D,Discount,173,-2868,-12879.630
15,S,SAMPLES,101,-95,-5991.110
1,ADJUST,Adjustment by john on 26/01/2010 16,38,2,1803.330
3,AMAZONFEE,AMAZON FEE,36,-30,-221520.500
2,ADJUST,Adjustment by john on 26/01/2010 17,26,6,5762.960
9,DCGSSGIRL,GIRLS PARTY BAG,23,92,295.630
7,DCGSSBOY,BOYS PARTY BAG,21,77,251.350


In [61]:
admin_keywords = (
    r"postage|manual|discount|bank charge|carriage|"
    r"test|amazon|adjustment|commission|bad debt|sample"
)

admin_like = sales_working[
    sales_working["Description"].str.contains(
        admin_keywords,
        case=False,
        na=False,
        regex=True
    )
].copy()

admin_summary = (
    admin_like
    .groupby(["StockCode", "Description"])
    .agg(
        Rows=("Invoice", "size"),
        TotalQuantity=("Quantity", "sum"),
        NetLineValue=("LineValue", "sum")
    )
    .reset_index()
    .sort_values("Rows", ascending=False)
)

admin_summary.head(50)

,StockCode,Description,Rows,TotalQuantity,NetLineValue
71,POST,POSTAGE,2079,5061,110430.410
69,DOT,DOTCOM POSTAGE,1423,1417,309844.100
70,M,Manual,1387,4193,-83326.330
66,C2,CARRIAGE,274,261,13136.000
68,D,Discount,173,-2868,-12879.630
72,S,SAMPLES,101,-95,-5991.110
65,BANK CHARGES,Bank Charges,94,-40,-33413.289
36,23444,Next Day Carriage,80,78,1185.000
35,23099,FRENCH CARRIAGE LANTERN,63,129,872.150
55,85168B,BLACK BAROQUE CARRIAGE CLOCK,52,72,652.170


In [62]:
r"[A-Za-z]+"

'[A-Za-z]+'

In [63]:
known_special_codes = [
    "POST",
    "DOT",
    "M",
    "m",
    "D",
    "S",
    "ADJUST",
    "A.DJUST",
    "ADJUST2",
    "AMAZONFEE",
    "CRUK",
    "B",
    "BANK CHARGES",
    "C2",
    "TEST001",
    "TEST002"
]

special_exact = sales_working[
    sales_working["StockCode"].isin(known_special_codes)
]

special_exact_summary = (
    special_exact
    .groupby(
        ["StockCode", "Description"],
        dropna=False
    )
    .agg(
        Rows=("Invoice", "size"),
        TotalQuantity=("Quantity", "sum"),
        NetLineValue=("LineValue", "sum")
    )
    .reset_index()
    .sort_values("Rows", ascending=False)
)

special_exact_summary

,StockCode,Description,Rows,TotalQuantity,NetLineValue
13,POST,POSTAGE,2079,5061,110430.410
11,DOT,DOTCOM POSTAGE,1423,1417,309844.100
12,M,Manual,1387,4193,-83326.330
8,C2,CARRIAGE,274,261,13136.000
10,D,Discount,173,-2868,-12879.630
14,S,SAMPLES,101,-95,-5991.110
7,BANK CHARGES,Bank Charges,94,-40,-33413.289
1,ADJUST,Adjustment by john on 26/01/2010 16,38,2,1803.330
4,AMAZONFEE,AMAZON FEE,36,-30,-221520.500
2,ADJUST,Adjustment by john on 26/01/2010 17,26,6,5762.960


### Line Category Classification

Known administrative and non-merchandise StockCodes are classified separately from physical merchandise. This allows financial and operational transaction lines to be retained for traceability while preventing postage, discounts, fees, adjustments, samples, and test records from distorting product-level KPIs.

In [64]:
# Normalize StockCode only for classification purposes
sales_working["StockCodeNormalized"] = (
    sales_working["StockCode"]
    .str.upper()
)

line_category_map = {
    "POST": "Shipping/Postage",
    "DOT": "Shipping/Postage",
    "C2": "Shipping/Postage",

    "D": "Discount",

    "M": "Manual/Financial Adjustment",
    "B": "Manual/Financial Adjustment",
    "BANK CHARGES": "Manual/Financial Adjustment",

    "AMAZONFEE": "Marketplace/Commission Fee",
    "CRUK": "Marketplace/Commission Fee",

    "ADJUST": "Operational Adjustment",
    "A.DJUST": "Operational Adjustment",
    "ADJUST2": "Operational Adjustment",

    "S": "Sample",

    "TEST001": "Test Record",
    "TEST002": "Test Record"
}

sales_working["LineCategory"] = (
    sales_working["StockCodeNormalized"]
    .map(line_category_map)
    .fillna("Merchandise")
)

In [65]:
sales_working["LineCategory"].value_counts()

LineCategory
Merchandise                    1023075
Shipping/Postage                  3776
Manual/Financial Adjustment       1498
Discount                           173
Sample                             101
Operational Adjustment              70
Marketplace/Commission Fee          52
Test Record                         16
Name: count, dtype: int64

In [66]:
line_category_summary = (
    sales_working
    .groupby("LineCategory")
    .agg(
        Rows=("Invoice", "size"),
        TotalQuantity=("Quantity", "sum"),
        NetLineValue=("LineValue", "sum")
    )
    .sort_values("Rows", ascending=False)
)

line_category_summary

,Rows,TotalQuantity,NetLineValue
LineCategory,,,
Merchandise,1023075,10476066,1.892909e+07
Shipping/Postage,3776,6739,4.334105e+05
Manual/Financial Adjustment,1498,4162,-2.664076e+05
Discount,173,-2868,-1.287963e+04
Sample,101,-95,-5.991110e+03
Operational Adjustment,70,8,7.566290e+03
Marketplace/Commission Fee,52,-46,-2.294539e+05
Test Record,16,56,2.035000e+02


In [67]:
pd.crosstab(
    sales_working["LineCategory"],
    sales_working["TransactionType"]
)

TransactionType,Cancellation/Return,Manual/Exceptional Adjustment,Operational Adjustment,Sale,Zero-Price/Non-Revenue
LineCategory,,,,,
Discount,168,0,0,5,0
Manual/Financial Adjustment,599,1,0,891,7
Marketplace/Commission Fee,49,0,0,3,0
Merchandise,17916,0,760,1003434,965
Operational Adjustment,31,0,0,39,0
Sample,98,0,0,3,0
Shipping/Postage,238,0,0,3533,5
Test Record,4,0,0,10,2


## Analysis-Ready Business Metrics

Derived fields are created to support consistent KPI calculations across Python, SQL, Excel, and Tableau.

Merchandise sales, customer returns, units sold, returned units, and net merchandise revenue are calculated separately from shipping, administrative charges, operational adjustments, and other non-merchandise activity.

In [68]:
# Gross merchandise sales
sales_working["GrossSales"] = 0.0

sales_working.loc[
    (sales_working["TransactionType"] == "Sale")
    & (sales_working["LineCategory"] == "Merchandise"),
    "GrossSales"
] = sales_working["LineValue"]


# Merchandise return value stored as a positive amount
sales_working["ReturnValue"] = 0.0

sales_working.loc[
    (sales_working["TransactionType"] == "Cancellation/Return")
    & (sales_working["LineCategory"] == "Merchandise"),
    "ReturnValue"
] = -sales_working["LineValue"]


# Net merchandise revenue
sales_working["NetMerchandiseRevenue"] = (
    sales_working["GrossSales"]
    - sales_working["ReturnValue"]
)


# Units actually sold
sales_working["UnitsSold"] = 0

sales_working.loc[
    (sales_working["TransactionType"] == "Sale")
    & (sales_working["LineCategory"] == "Merchandise"),
    "UnitsSold"
] = sales_working["Quantity"]


# Units returned
sales_working["ReturnedUnits"] = 0

sales_working.loc[
    (sales_working["TransactionType"] == "Cancellation/Return")
    & (sales_working["LineCategory"] == "Merchandise"),
    "ReturnedUnits"
] = -sales_working["Quantity"]

In [69]:
-sales_working["LineValue"]

0          -83.40
1          -81.00
2          -81.00
3         -100.80
4          -30.00
            ...  
1044843    -12.60
1044844    -16.60
1044845    -16.60
1044846    -14.85
1044847    -18.00
Name: LineValue, Length: 1028761, dtype: float64

In [70]:
gross_sales = sales_working["GrossSales"].sum()
return_value = sales_working["ReturnValue"].sum()
net_revenue = sales_working["NetMerchandiseRevenue"].sum()

units_sold = sales_working["UnitsSold"].sum()
returned_units = sales_working["ReturnedUnits"].sum()

print("Gross Merchandise Sales:", round(gross_sales, 2))
print("Return Value:", round(return_value, 2))
print("Net Merchandise Revenue:", round(net_revenue, 2))
print("Units Sold:", units_sold)
print("Returned Units:", returned_units)

Gross Merchandise Sales: 19645617.81
Return Value: 716532.13
Net Merchandise Revenue: 18929085.68
Units Sold: 11188141
Returned Units: 467741


In [71]:
revenue_return_rate = (
    return_value / gross_sales * 100
)

unit_return_rate = (
    returned_units / units_sold * 100
)

print(
    "Revenue Return Rate:",
    round(revenue_return_rate, 2),
    "%"
)

print(
    "Unit Return Rate:",
    round(unit_return_rate, 2),
    "%"
)

Revenue Return Rate: 3.65 %
Unit Return Rate: 4.18 %


## Core Business Summary

Core business metrics are calculated using completed merchandise sales. 
Orders represent unique non-cancellation invoices containing paid merchandise, while customer metrics use only transactions with an identifiable Customer ID.

In [72]:
merchandise_sales = sales_working[
    (sales_working["TransactionType"] == "Sale")
    & (sales_working["LineCategory"] == "Merchandise")
].copy()

In [73]:
total_orders = merchandise_sales["Invoice"].nunique()

known_customer_sales = merchandise_sales[
    merchandise_sales["Customer ID"].notna()
]

unique_customers = known_customer_sales["Customer ID"].nunique()

print("Completed Orders:", total_orders)
print("Identified Customers:", unique_customers)

Completed Orders: 39516
Identified Customers: 5852


In [74]:
average_order_value = gross_sales / total_orders

print(
    "Average Order Value:",
    round(average_order_value, 2)
)

Average Order Value: 497.16


In [75]:
country_summary = (
    merchandise_sales
    .groupby("Country")
    .agg(
        Orders=("Invoice", "nunique"),
        UnitsSold=("UnitsSold", "sum"),
        GrossSales=("GrossSales", "sum")
    )
    .sort_values("GrossSales", ascending=False)
)

print("Number of Countries:", merchandise_sales["Country"].nunique())

country_summary.head(15)

Number of Countries: 43


,Orders,UnitsSold,GrossSales
Country,,,
United Kingdom,36184,9176270,1.680278e+07
EIRE,581,336088,6.234142e+05
Netherlands,216,383625,5.497734e+05
Germany,753,223089,3.832890e+05
France,598,270582,3.110903e+05
Australia,89,103753,1.678000e+05
Spain,144,49996,9.776675e+04
Switzerland,85,52612,9.402459e+04
Sweden,99,88537,8.631914e+04


In [76]:
analysis_start = merchandise_sales["InvoiceDate"].min()
analysis_end = merchandise_sales["InvoiceDate"].max()

print("Analysis Start Date:", analysis_start)
print("Analysis End Date:", analysis_end)

Analysis Start Date: 2009-12-01 07:45:00
Analysis End Date: 2011-12-09 12:50:00


## Date and Time Feature Engineering

Calendar and time-based fields are derived from transaction timestamps to support monthly trends, year-over-year comparisons, seasonal analysis, weekday patterns, hourly demand analysis, and interactive dashboard filtering.

In [77]:
sales_working["Year"] = sales_working["InvoiceDate"].dt.year
sales_working["MonthNumber"] = sales_working["InvoiceDate"].dt.month
sales_working["Month"] = sales_working["InvoiceDate"].dt.month_name()
sales_working["Quarter"] = sales_working["InvoiceDate"].dt.quarter
sales_working["Weekday"] = sales_working["InvoiceDate"].dt.day_name()
sales_working["Hour"] = sales_working["InvoiceDate"].dt.hour

sales_working["YearMonth"] = (
    sales_working["InvoiceDate"]
    .dt.to_period("M")
    .astype(str)
)

In [78]:
sales_working[
    [
        "InvoiceDate",
        "Year",
        "MonthNumber",
        "Month",
        "Quarter",
        "Weekday",
        "Hour",
        "YearMonth"
    ]
].head()

,InvoiceDate,Year,MonthNumber,Month,Quarter,Weekday,Hour,YearMonth
0,2009-12-01 07:45:00,2009,12,December,4,Tuesday,7,2009-12
1,2009-12-01 07:45:00,2009,12,December,4,Tuesday,7,2009-12
2,2009-12-01 07:45:00,2009,12,December,4,Tuesday,7,2009-12
3,2009-12-01 07:45:00,2009,12,December,4,Tuesday,7,2009-12
4,2009-12-01 07:45:00,2009,12,December,4,Tuesday,7,2009-12


In [79]:
monthly_sales = (
    sales_working[
        (sales_working["TransactionType"] == "Sale")
        & (sales_working["LineCategory"] == "Merchandise")
    ]
    .groupby("YearMonth")
    .agg(
        GrossSales=("GrossSales", "sum"),
        Orders=("Invoice", "nunique"),
        UnitsSold=("UnitsSold", "sum")
    )
    .reset_index()
)

monthly_sales.head(10)

,YearMonth,GrossSales,Orders,UnitsSold
0,2009-12,798118.530,1666,425226
1,2010-01,612365.502,1049,390417
2,2010-02,537926.696,1189,381627
3,2010-03,761748.531,1647,525046
4,2010-04,646451.562,1435,366072
5,2010-05,643585.640,1484,395442
6,2010-06,696651.270,1612,406240
7,2010-07,633079.420,1507,337560
8,2010-08,674192.890,1402,471939
9,2010-09,869277.161,1789,583369


In [80]:
monthly_sales.sort_values(
    "GrossSales",
    ascending=False
).head(10)

,YearMonth,GrossSales,Orders,UnitsSold
23,2011-11,1452115.980,2751,746952
11,2010-11,1429751.842,2719,724268
22,2011-10,1103330.920,2005,619598
10,2010-10,1094563.630,2244,619515
21,2011-09,1028345.381,1818,568702
9,2010-09,869277.161,1789,583369
0,2009-12,798118.530,1666,425226
12,2010-12,775714.950,1550,357531
3,2010-03,761748.531,1647,525046
17,2011-05,740036.330,1668,394657


## Revenue Trend and Seasonality Analysis

Monthly merchandise performance is analyzed to identify growth patterns, seasonal peaks, and year-over-year changes. Partial periods are treated carefully to avoid misleading comparisons, particularly December 2011, which contains data only through December 9.

In [81]:
sales_working["MonthStart"] = (
    sales_working["InvoiceDate"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

In [82]:
monthly_performance = (
    sales_working
    .groupby("MonthStart")
    .agg(
        GrossSales=("GrossSales", "sum"),
        ReturnValue=("ReturnValue", "sum"),
        NetRevenue=("NetMerchandiseRevenue", "sum"),
        UnitsSold=("UnitsSold", "sum"),
        ReturnedUnits=("ReturnedUnits", "sum")
    )
    .reset_index()
)

monthly_performance["ReturnRate"] = (
    monthly_performance["ReturnValue"]
    / monthly_performance["GrossSales"]
    * 100
)

monthly_performance.head()

,MonthStart,GrossSales,ReturnValue,NetRevenue,UnitsSold,ReturnedUnits,ReturnRate
0,2009-12-01,798118.530,19558.08,778560.450,425226,9982,2.450523
1,2010-01-01,612365.502,7535.19,604830.312,390417,3305,1.230505
2,2010-02-01,537926.696,12575.44,525351.256,381627,6739,2.337761
3,2010-03-01,761748.531,9603.85,752144.681,525046,4257,1.260764
4,2010-04-01,646451.562,10025.26,636426.302,366072,5998,1.550814


In [83]:
monthly_orders = (
    merchandise_sales
    .groupby(
        merchandise_sales["InvoiceDate"]
        .dt.to_period("M")
        .dt.to_timestamp()
    )["Invoice"]
    .nunique()
    .reset_index(name="Orders")
    .rename(columns={"InvoiceDate": "MonthStart"})
)

monthly_performance = monthly_performance.merge(
    monthly_orders,
    on="MonthStart",
    how="left"
)

monthly_performance.head()

,MonthStart,GrossSales,ReturnValue,NetRevenue,UnitsSold,ReturnedUnits,ReturnRate,Orders
0,2009-12-01,798118.530,19558.08,778560.450,425226,9982,2.450523,1666
1,2010-01-01,612365.502,7535.19,604830.312,390417,3305,1.230505,1049
2,2010-02-01,537926.696,12575.44,525351.256,381627,6739,2.337761,1189
3,2010-03-01,761748.531,9603.85,752144.681,525046,4257,1.260764,1647
4,2010-04-01,646451.562,10025.26,636426.302,366072,5998,1.550814,1435


In [84]:
monthly_performance.sort_values(
    "NetRevenue",
    ascending=False
).head(10)

,MonthStart,GrossSales,ReturnValue,NetRevenue,UnitsSold,ReturnedUnits,ReturnRate,Orders
23,2011-11-01,1452115.980,24991.44,1427124.540,746952,12306,1.721036,2751
11,2010-11-01,1429751.842,35098.98,1394652.862,724268,17636,2.454900,2719
10,2010-10-01,1094563.630,25451.07,1069112.560,619515,11401,2.325225,2244
22,2011-10-01,1103330.920,41613.03,1061717.890,619598,22258,3.771582,2005
21,2011-09-01,1028345.381,16980.64,1011364.741,568702,6985,1.651258,1818
9,2010-09-01,869277.161,27686.42,841590.741,583369,94247,3.184993,1789
0,2009-12-01,798118.530,19558.08,778560.450,425226,9982,2.450523,1666
12,2010-12-01,775714.950,17547.86,758167.090,357531,15977,2.262153,1550
3,2010-03-01,761748.531,9603.85,752144.681,525046,4257,1.260764,1647
17,2011-05-01,740036.330,8948.23,731088.100,394657,4057,1.209161,1668


In [86]:
merchandise_sales = sales_working[
    (sales_working["TransactionType"] == "Sale")
    & (sales_working["LineCategory"] == "Merchandise")
].copy()

In [87]:
merchandise_sales[
    [
        "InvoiceDate",
        "Year",
        "Month",
        "Quarter",
        "Weekday",
        "YearMonth"
    ]
].head()

,InvoiceDate,Year,Month,Quarter,Weekday,YearMonth
0,2009-12-01 07:45:00,2009,December,4,Tuesday,2009-12
1,2009-12-01 07:45:00,2009,December,4,Tuesday,2009-12
2,2009-12-01 07:45:00,2009,December,4,Tuesday,2009-12
3,2009-12-01 07:45:00,2009,December,4,Tuesday,2009-12
4,2009-12-01 07:45:00,2009,December,4,Tuesday,2009-12


In [88]:
comparable_sales = merchandise_sales[
    (
        (merchandise_sales["InvoiceDate"] >= "2010-01-01")
        & (merchandise_sales["InvoiceDate"] < "2010-12-01")
    )
    |
    (
        (merchandise_sales["InvoiceDate"] >= "2011-01-01")
        & (merchandise_sales["InvoiceDate"] < "2011-12-01")
    )
].copy()

In [89]:
year_comparison = (
    comparable_sales
    .groupby("Year")
    .agg(
        GrossSales=("GrossSales", "sum"),
        Orders=("Invoice", "nunique"),
        UnitsSold=("UnitsSold", "sum")
    )
)

year_comparison

,GrossSales,Orders,UnitsSold
Year,,,
2010,8599594.144,18077,5201495
2011,8857690.933,17407,4891241


In [90]:
sales_growth = (
    (
        year_comparison.loc[2011, "GrossSales"]
        - year_comparison.loc[2010, "GrossSales"]
    )
    / year_comparison.loc[2010, "GrossSales"]
    * 100
)

order_growth = (
    (
        year_comparison.loc[2011, "Orders"]
        - year_comparison.loc[2010, "Orders"]
    )
    / year_comparison.loc[2010, "Orders"]
    * 100
)

units_growth = (
    (
        year_comparison.loc[2011, "UnitsSold"]
        - year_comparison.loc[2010, "UnitsSold"]
    )
    / year_comparison.loc[2010, "UnitsSold"]
    * 100
)

print("Jan-Nov Gross Sales Growth:", round(sales_growth, 2), "%")
print("Jan-Nov Order Growth:", round(order_growth, 2), "%")
print("Jan-Nov Units Growth:", round(units_growth, 2), "%")

Jan-Nov Gross Sales Growth: 3.0 %
Jan-Nov Order Growth: -3.71 %
Jan-Nov Units Growth: -5.96 %


### Order Value Analysis

Although Jan–Nov 2011 generated higher merchandise sales than the comparable 2010 period, both order volume and units sold declined. Average order value and revenue per unit are examined to understand the drivers behind this growth.

In [91]:
year_comparison["AverageOrderValue"] = (
    year_comparison["GrossSales"]
    / year_comparison["Orders"]
)

year_comparison["RevenuePerUnit"] = (
    year_comparison["GrossSales"]
    / year_comparison["UnitsSold"]
)

year_comparison

,GrossSales,Orders,UnitsSold,AverageOrderValue,RevenuePerUnit
Year,,,,,
2010,8599594.144,18077,5201495,475.720205,1.653293
2011,8857690.933,17407,4891241,508.857984,1.810929


In [92]:
aov_growth = (
    (
        year_comparison.loc[2011, "AverageOrderValue"]
        - year_comparison.loc[2010, "AverageOrderValue"]
    )
    / year_comparison.loc[2010, "AverageOrderValue"]
    * 100
)

revenue_per_unit_growth = (
    (
        year_comparison.loc[2011, "RevenuePerUnit"]
        - year_comparison.loc[2010, "RevenuePerUnit"]
    )
    / year_comparison.loc[2010, "RevenuePerUnit"]
    * 100
)

print("AOV Growth:", round(aov_growth, 2), "%")
print(
    "Revenue per Unit Growth:",
    round(revenue_per_unit_growth, 2),
    "%"
)

AOV Growth: 6.97 %
Revenue per Unit Growth: 9.53 %


### Monthly Seasonality

Monthly sales performance is compared across complete calendar months to identify recurring seasonal patterns. December 2011 is excluded from direct monthly comparison because the dataset ends on December 9, 2011.

In [93]:
seasonality_sales = merchandise_sales[
    (
        merchandise_sales["Year"].isin([2010, 2011])
    )
    &
    (
        merchandise_sales["MonthNumber"] <= 11
    )
].copy()

In [94]:
monthly_yoy = (
    seasonality_sales
    .groupby(["Year", "MonthNumber", "Month"])
    .agg(
        GrossSales=("GrossSales", "sum"),
        Orders=("Invoice", "nunique"),
        UnitsSold=("UnitsSold", "sum")
    )
    .reset_index()
    .sort_values(["Year", "MonthNumber"])
)

monthly_yoy

,Year,MonthNumber,Month,GrossSales,Orders,UnitsSold
0,2010,1,January,612365.502,1049,390417
1,2010,2,February,537926.696,1189,381627
2,2010,3,March,761748.531,1647,525046
3,2010,4,April,646451.562,1435,366072
4,2010,5,May,643585.640,1484,395442
5,2010,6,June,696651.270,1612,406240
6,2010,7,July,633079.420,1507,337560
7,2010,8,August,674192.890,1402,471939
8,2010,9,September,869277.161,1789,583369
9,2010,10,October,1094563.630,2244,619515


In [95]:
monthly_yoy["AverageOrderValue"] = (
    monthly_yoy["GrossSales"]
    / monthly_yoy["Orders"]
)

monthly_yoy

,Year,MonthNumber,Month,GrossSales,Orders,UnitsSold,AverageOrderValue
0,2010,1,January,612365.502,1049,390417,583.761203
1,2010,2,February,537926.696,1189,381627,452.419425
2,2010,3,March,761748.531,1647,525046,462.506698
3,2010,4,April,646451.562,1435,366072,450.488893
4,2010,5,May,643585.640,1484,395442,433.683046
5,2010,6,June,696651.270,1612,406240,432.165800
6,2010,7,July,633079.420,1507,337560,420.092515
7,2010,8,August,674192.890,1402,471939,480.879379
8,2010,9,September,869277.161,1789,583369,485.901152
9,2010,10,October,1094563.630,2244,619515,487.773454


In [96]:
monthly_comparison = monthly_yoy.pivot(
    index=["MonthNumber", "Month"],
    columns="Year",
    values="GrossSales"
).reset_index()

monthly_comparison

Year,MonthNumber,Month,2010,2011
0,1,January,612365.502,670439.460
1,2,February,537926.696,507866.540
2,3,March,761748.531,689841.840
3,4,April,646451.562,515469.661
4,5,May,643585.640,740036.330
5,6,June,696651.270,737683.990
6,7,July,633079.420,688252.671
7,8,August,674192.890,724308.160
8,9,September,869277.161,1028345.381
9,10,October,1094563.630,1103330.920


In [97]:
monthly_comparison["YoYGrowthPct"] = (
    (
        monthly_comparison[2011]
        - monthly_comparison[2010]
    )
    / monthly_comparison[2010]
    * 100
)

monthly_comparison

Year,MonthNumber,Month,2010,2011,YoYGrowthPct
0,1,January,612365.502,670439.460,9.483545
1,2,February,537926.696,507866.540,-5.588151
2,3,March,761748.531,689841.840,-9.439689
3,4,April,646451.562,515469.661,-20.261673
4,5,May,643585.640,740036.330,14.986458
5,6,June,696651.270,737683.990,5.889994
6,7,July,633079.420,688252.671,8.715060
7,8,August,674192.890,724308.160,7.433373
8,9,September,869277.161,1028345.381,18.298907
9,10,October,1094563.630,1103330.920,0.800985


In [98]:
monthly_comparison.sort_values(
    "YoYGrowthPct",
    ascending=False
)

Year,MonthNumber,Month,2010,2011,YoYGrowthPct
8,9,September,869277.161,1028345.381,18.298907
4,5,May,643585.640,740036.330,14.986458
0,1,January,612365.502,670439.460,9.483545
6,7,July,633079.420,688252.671,8.715060
7,8,August,674192.890,724308.160,7.433373
5,6,June,696651.270,737683.990,5.889994
10,11,November,1429751.842,1452115.980,1.564197
9,10,October,1094563.630,1103330.920,0.800985
1,2,February,537926.696,507866.540,-5.588151
2,3,March,761748.531,689841.840,-9.439689


### Seasonality Findings

Sales demonstrate a clear late-year seasonal pattern, with September through November consistently generating some of the highest merchandise revenue in both 2010 and 2011.

November was the highest-revenue month in both comparable years, generating approximately £1.43M in 2010 and £1.45M in 2011.

Year-over-year performance was mixed rather than uniformly positive. September 2011 recorded the strongest improvement at approximately 18.3%, followed by May at 15.0%. In contrast, April declined by approximately 20.3%, while March and February also underperformed their 2010 levels.

These results indicate that overall 2011 growth was supported by stronger performance during selected months, particularly around the late-year sales period.

## Product Performance Analysis

Merchandise products are evaluated by revenue, sales volume, order frequency, and returns to identify top-performing products and potential product-level risks.

In [99]:
product_sales = (
    merchandise_sales
    .groupby(["StockCode", "Description"])
    .agg(
        GrossSales=("GrossSales", "sum"),
        UnitsSold=("UnitsSold", "sum"),
        Orders=("Invoice", "nunique")
    )
    .reset_index()
)

print("Unique merchandise products:", product_sales["StockCode"].nunique())

product_sales.head()

Unique merchandise products: 4903


,StockCode,Description,GrossSales,UnitsSold,Orders
0,10002,INFLATABLE POLITICAL GLOBE,6942.26,8671,362
1,10002R,ROBOT PENCIL SHARPNER,20.57,4,3
2,10080,GROOVY CACTUS INFLATABLE,129.29,315,27
3,10109,BENDY COLOUR PENCILS,1.68,4,1
4,10120,DOGGY RUBBER,142.74,664,73


In [100]:
top_products_revenue = (
    product_sales
    .sort_values("GrossSales", ascending=False)
    .head(10)
)

top_products_revenue

,StockCode,Description,GrossSales,UnitsSold,Orders
1891,22423,REGENCY CAKESTAND 3 TIER,330590.32,26478,3918
4999,85123A,WHITE HANGING HEART T-LIGHT HOLDER,257546.20,94142,5356
3355,23843,"PAPER CRAFT , LITTLE BIRDIE",168469.60,80995,1
3716,47566,PARTY BUNTING,148318.28,28200,2674
4967,85099B,JUMBO BAG RED RETROSPOT,145961.83,77280,3245
4640,84879,ASSORTED COLOUR BIRD ORNAMENT,129324.49,80082,2807
1477,22086,PAPER CHAIN KIT 50'S CHRISTMAS,117760.29,35084,2018
2810,23166,MEDIUM CERAMIC TOP STORAGE JAR,81700.92,78033,247
4058,79321,CHILLI LIGHTS,80540.88,15841,1135
4227,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,71300.40,31409,756


In [101]:
top_products_units = (
    product_sales
    .sort_values("UnitsSold", ascending=False)
    .head(10)
)

top_products_units

,StockCode,Description,GrossSales,UnitsSold,Orders
4171,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,24445.61,106139,1019
4999,85123A,WHITE HANGING HEART T-LIGHT HOLDER,257546.20,94142,5356
3355,23843,"PAPER CRAFT , LITTLE BIRDIE",168469.60,80995,1
4640,84879,ASSORTED COLOUR BIRD ORNAMENT,129324.49,80082,2807
2810,23166,MEDIUM CERAMIC TOP STORAGE JAR,81700.92,78033,247
4967,85099B,JUMBO BAG RED RETROSPOT,145961.83,77280,3245
121,17003,BROCADE RING PURSE,14766.42,70369,456
1384,21977,PACK OF 60 PINK PAISLEY CAKE CASES,28081.73,56061,1993
4801,84991,60 TEATIME FAIRY CAKE CASES,27041.21,54028,2127
1607,22197,SMALL POPCORN HOLDER,42699.89,48561,1407


In [102]:
top_products_orders = (
    product_sales
    .sort_values("Orders", ascending=False)
    .head(10)
)

top_products_orders

,StockCode,Description,GrossSales,UnitsSold,Orders
4999,85123A,WHITE HANGING HEART T-LIGHT HOLDER,257546.20,94142,5356
1891,22423,REGENCY CAKESTAND 3 TIER,330590.32,26478,3918
4967,85099B,JUMBO BAG RED RETROSPOT,145961.83,77280,3245
4640,84879,ASSORTED COLOUR BIRD ORNAMENT,129324.49,80082,2807
3716,47566,PARTY BUNTING,148318.28,28200,2674
289,20727,LUNCH BAG BLACK SKULL.,45655.27,26678,2351
1357,21931,JUMBO STORAGE BAG SUKI,61537.55,29182,2329
693,21232,STRAWBERRY CERAMIC TRINKET BOX,47169.35,36628,2310
1875,22411,JUMBO SHOPPER VINTAGE RED PAISLEY,57318.71,27281,2192
1948,22469,HEART OF WICKER SMALL,48935.11,27908,2151


### High-Volume Transaction Investigation

Products with unusually high unit volumes are reviewed at the invoice level to determine whether rankings are being driven by legitimate bulk purchases, data-entry anomalies, or exceptional transactions.

Extreme transactions are not removed automatically. Their context is evaluated before deciding how they should be represented in product-performance analysis.

In [103]:
merchandise_sales[
    merchandise_sales["StockCode"] == "23843"
][
    [
        "Invoice",
        "StockCode",
        "Description",
        "Quantity",
        "Price",
        "GrossSales",
        "InvoiceDate",
        "Customer ID",
        "Country"
    ]
].sort_values(
    "Quantity",
    ascending=False
)

,Invoice,StockCode,Description,Quantity,Price,GrossSales,InvoiceDate,Customer ID,Country
1043359,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2.08,168469.6,2011-12-09 09:15:00,16446,United Kingdom


In [104]:
merchandise_sales[
    merchandise_sales["StockCode"] == "23166"
][
    [
        "Invoice",
        "StockCode",
        "Description",
        "Quantity",
        "Price",
        "GrossSales",
        "InvoiceDate",
        "Customer ID",
        "Country"
    ]
].sort_values(
    "Quantity",
    ascending=False
).head(20)

,Invoice,StockCode,Description,Quantity,Price,GrossSales,InvoiceDate,Customer ID,Country
564557,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,1.04,77183.60,2011-01-18 10:01:00,12346,United Kingdom
788305,561901,23166,MEDIUM CERAMIC TOP STORAGE JAR,288,1.25,360.00,2011-07-31 15:42:00,14156,EIRE
697400,553607,23166,MEDIUM CERAMIC TOP STORAGE JAR,240,1.04,249.60,2011-05-18 10:47:00,16684,United Kingdom
779451,561051,23166,MEDIUM CERAMIC TOP STORAGE JAR,144,1.04,149.76,2011-07-24 13:11:00,16684,United Kingdom
689708,552882,23166,MEDIUM CERAMIC TOP STORAGE JAR,96,1.04,99.84,2011-05-12 10:10:00,14646,Netherlands
988713,577669,23166,MEDIUM CERAMIC TOP STORAGE JAR,96,1.04,99.84,2011-11-21 10:48:00,15567,United Kingdom
731098,556917,23166,MEDIUM CERAMIC TOP STORAGE JAR,96,1.04,99.84,2011-06-15 13:37:00,12415,Australia
704508,554307,23166,MEDIUM CERAMIC TOP STORAGE JAR,96,1.04,99.84,2011-05-23 14:59:00,12989,United Kingdom
807867,563614,23166,MEDIUM CERAMIC TOP STORAGE JAR,96,1.04,99.84,2011-08-18 08:51:00,12415,Australia
1029319,580665,23166,MEDIUM CERAMIC TOP STORAGE JAR,96,1.04,99.84,2011-12-05 14:06:00,16684,United Kingdom


In [105]:
largest_sale_lines = (
    merchandise_sales[
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "Price",
            "GrossSales",
            "InvoiceDate",
            "Customer ID",
            "Country"
        ]
    ]
    .sort_values(
        "Quantity",
        ascending=False
    )
    .head(25)
)

largest_sale_lines

,Invoice,StockCode,Description,Quantity,Price,GrossSales,InvoiceDate,Customer ID,Country
1043359,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2.08,168469.60,2011-12-09 09:15:00,16446,United Kingdom
564557,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,1.04,77183.60,2011-01-18 10:01:00,12346,United Kingdom
90857,497946,37410,BLACK AND WHITE PAISLEY FLOWER MUG,19152,0.10,1915.20,2010-02-15 11:57:00,13902,Denmark
127168,501534,21091,SET/6 WOODLAND PAPER PLATES,12960,0.10,1296.00,2010-03-17 13:09:00,13902,Denmark
127166,501534,21099,SET/6 STRAWBERRY PAPER CUPS,12960,0.10,1296.00,2010-03-17 13:09:00,13902,Denmark
127169,501534,21085,SET/6 WOODLAND PAPER CUPS,12744,0.10,1274.40,2010-03-17 13:09:00,13902,Denmark
127167,501534,21092,SET/6 STRAWBERRY PAPER PLATES,12480,0.10,1248.00,2010-03-17 13:09:00,13902,Denmark
135027,502269,21984,PACK OF 12 PINK PAISLEY TISSUES,10000,0.25,2500.00,2010-03-23 15:36:00,17940,United Kingdom
135028,502269,21982,PACK OF 12 SUKI TISSUES,10000,0.25,2500.00,2010-03-23 15:36:00,17940,United Kingdom
135029,502269,21980,PACK OF 12 RED SPOTTY TISSUES,10000,0.25,2500.00,2010-03-23 15:36:00,17940,United Kingdom


### Bulk Transaction Impact

Extreme-volume merchandise transactions are retained because their quantities, prices, and transaction values are internally consistent and may represent legitimate wholesale orders.

Their influence on product rankings is measured separately so that high-volume one-off purchases can be distinguished from products with broad, recurring customer demand.

In [107]:
product_invoice_sales = (
    merchandise_sales
    .groupby(
        ["StockCode", "Description", "Invoice"]
    )
    .agg(
        InvoiceUnits=("UnitsSold", "sum"),
        InvoiceRevenue=("GrossSales", "sum")
    )
    .reset_index()
)

largest_product_invoice = (
    product_invoice_sales
    .groupby(["StockCode", "Description"])
    .agg(
        LargestInvoiceUnits=("InvoiceUnits", "max"),
        LargestInvoiceRevenue=("InvoiceRevenue", "max")
    )
    .reset_index()
)

product_concentration = product_sales.merge(
    largest_product_invoice,
    on=["StockCode", "Description"],
    how="left"
)

product_concentration["LargestInvoiceUnitSharePct"] = (
    product_concentration["LargestInvoiceUnits"]
    / product_concentration["UnitsSold"]
    * 100
)

product_concentration["LargestInvoiceRevenueSharePct"] = (
    product_concentration["LargestInvoiceRevenue"]
    / product_concentration["GrossSales"]
    * 100
)

In [108]:
product_concentration[
    product_concentration["StockCode"].isin(
        ["23843", "23166", "85123A", "84077"]
    )
][
    [
        "StockCode",
        "Description",
        "GrossSales",
        "UnitsSold",
        "Orders",
        "LargestInvoiceUnits",
        "LargestInvoiceUnitSharePct",
        "LargestInvoiceRevenueSharePct"
    ]
]

,StockCode,Description,GrossSales,UnitsSold,Orders,LargestInvoiceUnits,LargestInvoiceUnitSharePct,LargestInvoiceRevenueSharePct
2810,23166,MEDIUM CERAMIC TOP STORAGE JAR,81700.92,78033,247,74215,95.107198,94.470907
3355,23843,"PAPER CRAFT , LITTLE BIRDIE",168469.60,80995,1,80995,100.000000,100.000000
4171,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,24445.61,106139,1019,4800,4.522372,4.123440
4998,85123A,CREAM HANGING HEART T-LIGHT HOLDER,178.51,61,9,32,52.459016,45.711725
4999,85123A,WHITE HANGING HEART T-LIGHT HOLDER,257546.20,94142,5356,1930,2.050095,1.910919


In [109]:
high_concentration_products = (
    product_concentration[
        (product_concentration["UnitsSold"] >= 1000)
        &
        (product_concentration["LargestInvoiceUnitSharePct"] >= 50)
    ]
    .sort_values(
        "LargestInvoiceUnitSharePct",
        ascending=False
    )
)

high_concentration_products.head(20)

,StockCode,Description,GrossSales,UnitsSold,Orders,LargestInvoiceUnits,LargestInvoiceRevenue,LargestInvoiceUnitSharePct,LargestInvoiceRevenueSharePct
3355,23843,"PAPER CRAFT , LITTLE BIRDIE",168469.60,80995,1,80995,168469.60,100.000000,100.000000
3534,37351,ORANGE FLOWER MUG,612.51,5404,11,5364,536.40,99.259808,87.574080
3535,37352,BIRD IN TREE MUG,504.55,4643,8,4608,460.80,99.246177,91.328907
49,16044,POP-ART FLUORESCENT PENS,389.22,6193,3,6144,368.64,99.208784,94.712502
3538,37379B,BLUE CHERRY BLOSSOM CUP & SAUCER,181.50,1122,3,1110,166.50,98.930481,91.735537
5140,85218,S/5 MINI ICE CREAM FRIDGE MAGNETS,625.20,1992,2,1968,590.40,98.795181,94.433781
352,20800,LARGE GLASS SUNDAE DISH CLEAR,303.96,1005,5,992,248.00,98.706468,81.589683
850,21392,RED POLKADOT PUDDING BOWL,608.90,3717,17,3648,474.24,98.143664,77.884710
3690,47503E,ASS FLORAL PRINT SCISSORS,1186.90,6842,25,6696,1004.40,97.866121,84.623810
3691,47503F,ASS FLORAL PRINT TORCH,729.95,2567,11,2504,626.00,97.545773,85.759299


### Product Return Performance

Merchandise returns are evaluated at the product level to identify products associated with high returned-unit volumes, high return values, and elevated return rates.

Return rates are interpreted together with sales volume so that products with very small sales bases are not treated as equivalent to high-volume products.

In [110]:
merchandise_returns = sales_working[
    (sales_working["TransactionType"] == "Cancellation/Return")
    & (sales_working["LineCategory"] == "Merchandise")
].copy()

print("Merchandise return rows:", len(merchandise_returns))
print("Returned units:", merchandise_returns["ReturnedUnits"].sum())
print("Return value:", round(merchandise_returns["ReturnValue"].sum(), 2))

Merchandise return rows: 17916
Returned units: 467741
Return value: 716532.13


In [111]:
product_returns = (
    merchandise_returns
    .groupby("StockCode")
    .agg(
        ReturnedUnits=("ReturnedUnits", "sum"),
        ReturnValue=("ReturnValue", "sum"),
        ReturnInvoices=("Invoice", "nunique")
    )
    .reset_index()
)

In [112]:
product_sales_by_code = (
    merchandise_sales
    .groupby("StockCode")
    .agg(
        GrossSales=("GrossSales", "sum"),
        UnitsSold=("UnitsSold", "sum"),
        Orders=("Invoice", "nunique")
    )
    .reset_index()
)

In [113]:
product_descriptions = (
    merchandise_sales
    .groupby("StockCode")["Description"]
    .agg(
        lambda x: x.mode().iloc[0]
        if not x.mode().empty
        else x.iloc[0]
    )
    .reset_index()
)

In [114]:
product_performance = (
    product_sales_by_code
    .merge(
        product_returns,
        on="StockCode",
        how="left"
    )
    .merge(
        product_descriptions,
        on="StockCode",
        how="left"
    )
)

product_performance[
    ["ReturnedUnits", "ReturnValue", "ReturnInvoices"]
] = product_performance[
    ["ReturnedUnits", "ReturnValue", "ReturnInvoices"]
].fillna(0)

In [115]:
product_performance["UnitReturnRatePct"] = (
    product_performance["ReturnedUnits"]
    / product_performance["UnitsSold"]
    * 100
)

product_performance["RevenueReturnRatePct"] = (
    product_performance["ReturnValue"]
    / product_performance["GrossSales"]
    * 100
)

product_performance["NetRevenue"] = (
    product_performance["GrossSales"]
    - product_performance["ReturnValue"]
)

In [116]:
product_performance.sort_values(
    "ReturnedUnits",
    ascending=False
)[
    [
        "StockCode",
        "Description",
        "UnitsSold",
        "ReturnedUnits",
        "UnitReturnRatePct",
        "GrossSales",
        "ReturnValue",
        "RevenueReturnRatePct",
        "Orders",
        "ReturnInvoices"
    ]
].head(15)

,StockCode,Description,UnitsSold,ReturnedUnits,UnitReturnRatePct,GrossSales,ReturnValue,RevenueReturnRatePct,Orders,ReturnInvoices
2798,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,80995.0,100.000000,168469.60,168469.60,100.000000,1,1.0
2353,23166,MEDIUM CERAMIC TOP STORAGE JAR,78033,74494.0,95.464739,81700.92,77479.64,94.833253,247,10.0
3594,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,31409,9381.0,29.867236,71300.40,330.45,0.463462,756,6.0
516,21088,SET/6 FRUIT SALAD PAPER CUPS,16615,7140.0,42.973217,1816.73,572.52,31.513764,136,2.0
523,21096,SET/6 FRUIT SALAD PAPER PLATES,15919,7008.0,44.022866,2711.71,911.04,33.596513,143,1.0
48,16047,POP ART PEN CASE & PENS,10612,5184.0,48.850358,1054.70,414.72,39.321134,31,1.0
4284,85110,BLACK SILVER FLOWER T-LIGHT HOLDER,11520,5040.0,43.750000,1138.04,387.36,34.037468,23,2.0
2966,37340,MULTICOLOUR SPRING FLOWER MUG,13665,4996.0,36.560556,3881.33,501.22,12.913615,204,4.0
47,16046,TEATIME PEN CASE & PENS,10056,4632.0,46.062053,1223.50,389.04,31.797303,59,2.0
4351,85160A,WHITE BIRD GARDEN DESIGN MUG,8816,4320.0,49.001815,1493.75,561.60,37.596653,32,1.0


In [117]:
product_performance.sort_values(
    "ReturnValue",
    ascending=False
)[
    [
        "StockCode",
        "Description",
        "GrossSales",
        "ReturnValue",
        "RevenueReturnRatePct",
        "UnitsSold",
        "ReturnedUnits",
        "UnitReturnRatePct"
    ]
].head(15)

,StockCode,Description,GrossSales,ReturnValue,RevenueReturnRatePct,UnitsSold,ReturnedUnits,UnitReturnRatePct
2798,23843,"PAPER CRAFT , LITTLE BIRDIE",168469.60,168469.60,100.000000,80995,80995.0,100.000000
2353,23166,MEDIUM CERAMIC TOP STORAGE JAR,81700.92,77479.64,94.833253,78033,74494.0,95.464739
1638,22423,REGENCY CAKESTAND 3 TIER,330590.32,16545.30,5.004774,26478,1450.0,5.476244
4301,85123A,WHITE HANGING HEART T-LIGHT HOLDER,257724.71,9387.10,3.642297,94203,3637.0,3.860811
3247,71477,COLOUR GLASS. STAR T-LIGHT HOLDER,44680.75,7164.12,16.034019,15089,2574.0,17.058785
531,21108,FAIRY CAKE FLANNEL ASSORTED COLOUR,25407.27,6614.37,26.033375,12962,3159.0,24.371239
3448,79323W,WHITE CHERRY LIGHTS,18048.72,6421.10,35.576484,3004,1088.0,36.218375
1148,21843,RED RETROSPOT CAKE STAND,64918.30,5270.25,8.118281,6128,587.0,9.578982
3543,84078A,SET/4 WHITE RETRO STORAGE CUBES,43844.96,5187.65,11.831805,1128,147.0,13.031915
2300,23113,PANTRY CHOPPING BOARD,5868.50,4803.06,81.844764,1154,946.0,81.975737


### Net Product Performance and Return Risk

Gross product rankings can be distorted by large transactions that are subsequently returned or cancelled. Net units and net revenue are therefore calculated alongside return rates to provide a more accurate representation of realized product performance.

Product return risk is evaluated together with sales volume and transaction concentration rather than using return percentage alone.

In [118]:
product_performance["NetUnits"] = (
    product_performance["UnitsSold"]
    - product_performance["ReturnedUnits"]
)

product_performance["NetRevenue"] = (
    product_performance["GrossSales"]
    - product_performance["ReturnValue"]
)

In [119]:
product_performance = product_performance.merge(
    product_concentration[
        [
            "StockCode",
            "LargestInvoiceUnitSharePct",
            "LargestInvoiceRevenueSharePct"
        ]
    ],
    on="StockCode",
    how="left"
)

In [120]:
product_performance["DemandPattern"] = "Broad/Recurring"

product_performance.loc[
    product_performance["LargestInvoiceUnitSharePct"] >= 50,
    "DemandPattern"
] = "Bulk-Order Concentrated"

In [121]:
product_performance[
    product_performance["StockCode"].isin(
        ["23843", "23166", "85123A", "84077", "22423"]
    )
][
    [
        "StockCode",
        "Description",
        "GrossSales",
        "ReturnValue",
        "NetRevenue",
        "UnitsSold",
        "ReturnedUnits",
        "NetUnits",
        "UnitReturnRatePct",
        "LargestInvoiceUnitSharePct",
        "DemandPattern"
    ]
]

,StockCode,Description,GrossSales,ReturnValue,NetRevenue,UnitsSold,ReturnedUnits,NetUnits,UnitReturnRatePct,LargestInvoiceUnitSharePct,DemandPattern
1891,22423,REGENCY CAKESTAND 3 TIER,330590.32,16545.30,314045.02,26478,1450.0,25028.0,5.476244,1.359619,Broad/Recurring
2810,23166,MEDIUM CERAMIC TOP STORAGE JAR,81700.92,77479.64,4221.28,78033,74494.0,3539.0,95.464739,95.107198,Bulk-Order Concentrated
3355,23843,"PAPER CRAFT , LITTLE BIRDIE",168469.60,168469.60,0.00,80995,80995.0,0.0,100.000000,100.000000,Bulk-Order Concentrated
4171,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,24445.61,361.68,24083.93,106139,1704.0,104435.0,1.605442,4.522372,Broad/Recurring
4998,85123A,WHITE HANGING HEART T-LIGHT HOLDER,257724.71,9387.10,248337.61,94203,3637.0,90566.0,3.860811,52.459016,Bulk-Order Concentrated
4999,85123A,WHITE HANGING HEART T-LIGHT HOLDER,257724.71,9387.10,248337.61,94203,3637.0,90566.0,3.860811,2.050095,Broad/Recurring


In [122]:
return_risk_products = (
    product_performance[
        (product_performance["UnitsSold"] >= 1000)
        & (product_performance["Orders"] >= 20)
    ]
    .sort_values(
        "UnitReturnRatePct",
        ascending=False
    )
)

return_risk_products[
    [
        "StockCode",
        "Description",
        "UnitsSold",
        "ReturnedUnits",
        "UnitReturnRatePct",
        "GrossSales",
        "ReturnValue",
        "NetRevenue",
        "Orders",
        "DemandPattern"
    ]
].head(20)

,StockCode,Description,UnitsSold,ReturnedUnits,UnitReturnRatePct,GrossSales,ReturnValue,NetRevenue,Orders,DemandPattern
2810,23166,MEDIUM CERAMIC TOP STORAGE JAR,78033,74494.0,95.464739,81700.92,77479.64,4221.28,247,Bulk-Order Concentrated
2748,23113,PANTRY CHOPPING BOARD,1154,946.0,81.975737,5868.50,4803.06,1065.44,57,Bulk-Order Concentrated
5051,85160A,WHITE BIRD GARDEN DESIGN MUG,8816,4320.0,49.001815,1493.75,561.60,932.15,32,Broad/Recurring
3551,37444B,BLUE BREAKFAST CUP AND SAUCER,2010,984.0,48.955224,409.60,127.92,281.68,28,Broad/Recurring
52,16047,POP ART PEN CASE & PENS,10612,5184.0,48.850358,1054.70,414.72,639.98,31,Broad/Recurring
4831,85006,SET 4 NURSERY DES ROUND BOXES,3049,1480.0,48.540505,1289.31,564.40,724.91,37,Broad/Recurring
5052,85160B,BLACK BIRD GARDEN DESIGN MUG,6155,2981.0,48.432169,1032.70,391.41,641.29,36,Broad/Recurring
851,21392,RED SPOTTY PUDDING BOWL,7545,3650.0,48.376408,1478.13,478.44,999.69,48,Bulk-Order Concentrated
850,21392,RED SPOTTY PUDDING BOWL,7545,3650.0,48.376408,1478.13,478.44,999.69,48,Bulk-Order Concentrated
3550,37444A,YELLOW BREAKFAST CUP AND SAUCER,2709,1309.0,48.320413,563.43,172.99,390.44,43,Broad/Recurring


### Product Concentration Validation

Product concentration is recalculated at the StockCode level to ensure each product has a single analytical record. Description variations are treated as product-label inconsistencies rather than separate products.

In [124]:
product_invoice_concentration = (
    merchandise_sales
    .groupby(["StockCode", "Invoice"])
    .agg(
        InvoiceUnits=("UnitsSold", "sum"),
        InvoiceRevenue=("GrossSales", "sum")
    )
    .reset_index()
)

In [125]:
product_concentration_by_code = (
    product_invoice_concentration
    .groupby("StockCode")
    .agg(
        LargestInvoiceUnits=("InvoiceUnits", "max"),
        LargestInvoiceRevenue=("InvoiceRevenue", "max")
    )
    .reset_index()
)

In [126]:
product_concentration_by_code = (
    product_sales_by_code
    .merge(
        product_concentration_by_code,
        on="StockCode",
        how="left"
    )
)

In [127]:
product_concentration_by_code["LargestInvoiceUnitSharePct"] = (
    product_concentration_by_code["LargestInvoiceUnits"]
    / product_concentration_by_code["UnitsSold"]
    * 100
)

product_concentration_by_code["LargestInvoiceRevenueSharePct"] = (
    product_concentration_by_code["LargestInvoiceRevenue"]
    / product_concentration_by_code["GrossSales"]
    * 100
)

In [128]:
product_performance = (
    product_sales_by_code
    .merge(
        product_returns,
        on="StockCode",
        how="left"
    )
    .merge(
        product_descriptions,
        on="StockCode",
        how="left"
    )
    .merge(
        product_concentration_by_code[
            [
                "StockCode",
                "LargestInvoiceUnitSharePct",
                "LargestInvoiceRevenueSharePct"
            ]
        ],
        on="StockCode",
        how="left"
    )
)

In [129]:
product_performance[
    ["ReturnedUnits", "ReturnValue", "ReturnInvoices"]
] = product_performance[
    ["ReturnedUnits", "ReturnValue", "ReturnInvoices"]
].fillna(0)

In [130]:
product_performance["UnitReturnRatePct"] = (
    product_performance["ReturnedUnits"]
    / product_performance["UnitsSold"]
    * 100
)

product_performance["RevenueReturnRatePct"] = (
    product_performance["ReturnValue"]
    / product_performance["GrossSales"]
    * 100
)

product_performance["NetUnits"] = (
    product_performance["UnitsSold"]
    - product_performance["ReturnedUnits"]
)

product_performance["NetRevenue"] = (
    product_performance["GrossSales"]
    - product_performance["ReturnValue"]
)

product_performance["DemandPattern"] = "Broad/Recurring"

product_performance.loc[
    product_performance["LargestInvoiceUnitSharePct"] >= 50,
    "DemandPattern"
] = "Bulk-Order Concentrated"

In [131]:
print("Rows in product_performance:", len(product_performance))
print(
    "Unique StockCodes:",
    product_performance["StockCode"].nunique()
)

print(
    "Duplicate StockCodes:",
    product_performance["StockCode"].duplicated().sum()
)

Rows in product_performance: 4903
Unique StockCodes: 4903
Duplicate StockCodes: 0


In [132]:
return_risk_products = (
    product_performance[
        (product_performance["UnitsSold"] >= 1000)
        & (product_performance["Orders"] >= 20)
    ]
    .sort_values(
        "UnitReturnRatePct",
        ascending=False
    )
)

In [133]:
return_risk_products[
    [
        "StockCode",
        "Description",
        "UnitsSold",
        "ReturnedUnits",
        "UnitReturnRatePct",
        "GrossSales",
        "ReturnValue",
        "NetRevenue",
        "Orders",
        "DemandPattern"
    ]
].head(20)

,StockCode,Description,UnitsSold,ReturnedUnits,UnitReturnRatePct,GrossSales,ReturnValue,NetRevenue,Orders,DemandPattern
2353,23166,MEDIUM CERAMIC TOP STORAGE JAR,78033,74494.0,95.464739,81700.92,77479.64,4221.28,247,Bulk-Order Concentrated
2300,23113,PANTRY CHOPPING BOARD,1154,946.0,81.975737,5868.50,4803.06,1065.44,57,Bulk-Order Concentrated
4351,85160A,WHITE BIRD GARDEN DESIGN MUG,8816,4320.0,49.001815,1493.75,561.60,932.15,32,Broad/Recurring
2988,37444B,BLUE BREAKFAST CUP AND SAUCER,2010,984.0,48.955224,409.60,127.92,281.68,28,Broad/Recurring
48,16047,POP ART PEN CASE & PENS,10612,5184.0,48.850358,1054.70,414.72,639.98,31,Broad/Recurring
4141,85006,SET 4 NURSERY DES ROUND BOXES,3049,1480.0,48.540505,1289.31,564.40,724.91,37,Broad/Recurring
4352,85160B,BLACK BIRD GARDEN DESIGN MUG,6155,2981.0,48.432169,1032.70,391.41,641.29,36,Broad/Recurring
771,21392,RED SPOTTY PUDDING BOWL,7545,3650.0,48.376408,1478.13,478.44,999.69,48,Broad/Recurring
2987,37444A,YELLOW BREAKFAST CUP AND SAUCER,2709,1309.0,48.320413,563.43,172.99,390.44,43,Broad/Recurring
3950,84847,FLORAL BATHROOM SET,2330,1116.0,47.896996,1149.73,368.28,781.45,49,Broad/Recurring


In [134]:
print("Rows in product_performance:", len(product_performance))
print("Unique StockCodes:", product_performance["StockCode"].nunique())
print("Duplicate StockCodes:", product_performance["StockCode"].duplicated().sum())

Rows in product_performance: 4903
Unique StockCodes: 4903
Duplicate StockCodes: 0


## Customer Performance Analysis

Customer-level analysis is restricted to transactions with an identifiable Customer ID.

Customers are evaluated using merchandise revenue, purchasing frequency, units purchased, average order value, and purchase activity over time. Anonymous transactions remain included in overall business metrics but are excluded from customer-specific analysis.

In [135]:
customer_sales = merchandise_sales[
    merchandise_sales["Customer ID"].notna()
].copy()

print("Known-customer merchandise rows:", len(customer_sales))
print("Unique identified customers:", customer_sales["Customer ID"].nunique())

Known-customer merchandise rows: 776596
Unique identified customers: 5852


In [136]:
customer_performance = (
    customer_sales
    .groupby("Customer ID")
    .agg(
        GrossSales=("GrossSales", "sum"),
        Orders=("Invoice", "nunique"),
        UnitsPurchased=("UnitsSold", "sum"),
        FirstPurchase=("InvoiceDate", "min"),
        LastPurchase=("InvoiceDate", "max"),
        Countries=("Country", "nunique")
    )
    .reset_index()
)

In [137]:
customer_performance["AverageOrderValue"] = (
    customer_performance["GrossSales"]
    / customer_performance["Orders"]
)

In [138]:
top_customers_revenue = (
    customer_performance
    .sort_values("GrossSales", ascending=False)
    .head(15)
)

top_customers_revenue

,Customer ID,GrossSales,Orders,UnitsPurchased,FirstPurchase,LastPurchase,Countries,AverageOrderValue
5666,18102,580987.04,145,181645,2009-12-01 09:24:00,2011-12-09 11:50:00,1,4006.807172
2262,14646,526751.52,145,367072,2009-12-02 16:52:00,2011-12-08 12:12:00,1,3632.769103
1778,14156,303069.88,144,164281,2009-12-01 12:30:00,2011-11-30 10:54:00,1,2104.651944
2520,14911,272252.79,373,147804,2009-12-01 11:41:00,2011-12-08 15:54:00,1,729.900241
5026,17450,244784.25,51,83914,2010-09-27 16:59:00,2011-12-01 13:29:00,1,4799.691176
1321,13694,195640.69,143,188201,2009-12-04 15:26:00,2011-12-06 09:32:00,1,1368.116713
5085,17511,172132.87,60,117174,2009-12-02 10:52:00,2011-12-07 10:12:00,1,2868.881167
4037,16446,168472.50,2,80997,2011-05-18 09:52:00,2011-12-09 09:15:00,1,84236.250000
4271,16684,147142.77,55,104810,2009-12-07 12:56:00,2011-12-05 14:06:00,1,2675.323091
67,12415,144033.37,24,91443,2010-06-30 08:30:00,2011-11-15 14:22:00,1,6001.390417


In [139]:
top_customers_orders = (
    customer_performance
    .sort_values("Orders", ascending=False)
    .head(15)
)

top_customers_orders

,Customer ID,GrossSales,Orders,UnitsPurchased,FirstPurchase,LastPurchase,Countries,AverageOrderValue
2520,14911,272252.79,373,147804,2009-12-01 11:41:00,2011-12-08 15:54:00,1,729.900241
393,12748,49176.74,322,36853,2009-12-04 17:31:00,2011-12-09 12:20:00,1,152.722795
5408,17841,68519.95,211,36565,2009-12-02 15:41:00,2011-12-08 12:07:00,1,324.739100
2916,15311,114671.42,207,69908,2009-12-01 11:21:00,2011-12-09 12:00:00,1,553.968213
730,13089,113416.91,203,58932,2009-12-02 15:44:00,2011-12-07 09:02:00,1,558.703990
2222,14606,29648.75,184,15195,2009-12-03 12:40:00,2011-12-08 19:28:00,1,161.134511
5416,17850,51208.87,155,21052,2009-12-05 12:28:00,2010-12-02 15:27:00,1,330.379806
5666,18102,580987.04,145,181645,2009-12-01 09:24:00,2011-12-09 11:50:00,1,4006.807172
2262,14646,526751.52,145,367072,2009-12-02 16:52:00,2011-12-08 12:12:00,1,3632.769103
1778,14156,303069.88,144,164281,2009-12-01 12:30:00,2011-11-30 10:54:00,1,2104.651944


In [140]:
total_known_customer_sales = customer_performance["GrossSales"].sum()

customer_performance = customer_performance.sort_values(
    "GrossSales",
    ascending=False
).reset_index(drop=True)

customer_performance["RevenueSharePct"] = (
    customer_performance["GrossSales"]
    / total_known_customer_sales
    * 100
)

customer_performance["CumulativeRevenueSharePct"] = (
    customer_performance["RevenueSharePct"].cumsum()
)

In [141]:
customer_performance[
    [
        "Customer ID",
        "GrossSales",
        "Orders",
        "UnitsPurchased",
        "AverageOrderValue",
        "RevenueSharePct",
        "CumulativeRevenueSharePct"
    ]
].head(20)

,Customer ID,GrossSales,Orders,UnitsPurchased,AverageOrderValue,RevenueSharePct,CumulativeRevenueSharePct
0,18102,580987.04,145,181645,4006.807172,3.403839,3.403839
1,14646,526751.52,145,367072,3632.769103,3.086088,6.489927
2,14156,303069.88,144,164281,2104.651944,1.775601,8.265528
3,14911,272252.79,373,147804,729.900241,1.595052,9.860580
4,17450,244784.25,51,83914,4799.691176,1.434122,11.294702
5,13694,195640.69,143,188201,1368.116713,1.146203,12.440905
6,17511,172132.87,60,117174,2868.881167,1.008478,13.449383
7,16446,168472.50,2,80997,84236.250000,0.987033,14.436416
8,16684,147142.77,55,104810,2675.323091,0.862068,15.298483
9,12415,144033.37,24,91443,6001.390417,0.843851,16.142334


### Net Customer Performance

Gross customer sales can be distorted by transactions that are subsequently returned or cancelled. Customer performance is therefore evaluated using both gross sales and merchandise returns to calculate realized net revenue and net units.

Anonymous transactions remain excluded because customer-level behavior cannot be reliably attributed without a Customer ID.

In [142]:
customer_returns = merchandise_returns[
    merchandise_returns["Customer ID"].notna()
].copy()

print("Known-customer return rows:", len(customer_returns))
print(
    "Customers with merchandise returns:",
    customer_returns["Customer ID"].nunique()
)

Known-customer return rows: 17587
Customers with merchandise returns: 2445


In [143]:
customer_return_summary = (
    customer_returns
    .groupby("Customer ID")
    .agg(
        ReturnValue=("ReturnValue", "sum"),
        ReturnedUnits=("ReturnedUnits", "sum"),
        ReturnInvoices=("Invoice", "nunique")
    )
    .reset_index()
)

In [144]:
customer_performance = customer_performance.merge(
    customer_return_summary,
    on="Customer ID",
    how="left"
)

customer_performance[
    ["ReturnValue", "ReturnedUnits", "ReturnInvoices"]
] = customer_performance[
    ["ReturnValue", "ReturnedUnits", "ReturnInvoices"]
].fillna(0)

In [145]:
customer_performance["NetRevenue"] = (
    customer_performance["GrossSales"]
    - customer_performance["ReturnValue"]
)

customer_performance["NetUnits"] = (
    customer_performance["UnitsPurchased"]
    - customer_performance["ReturnedUnits"]
)

customer_performance["RevenueReturnRatePct"] = (
    customer_performance["ReturnValue"]
    / customer_performance["GrossSales"]
    * 100
)

customer_performance["NetAverageOrderValue"] = (
    customer_performance["NetRevenue"]
    / customer_performance["Orders"]
)

In [146]:
customer_performance[
    customer_performance["Customer ID"].isin(
        ["16446", "12346", "18102", "14646"]
    )
][
    [
        "Customer ID",
        "GrossSales",
        "ReturnValue",
        "NetRevenue",
        "Orders",
        "UnitsPurchased",
        "ReturnedUnits",
        "NetUnits",
        "RevenueReturnRatePct",
        "AverageOrderValue",
        "NetAverageOrderValue"
    ]
]

,Customer ID,GrossSales,ReturnValue,NetRevenue,Orders,UnitsPurchased,ReturnedUnits,NetUnits,RevenueReturnRatePct,AverageOrderValue,NetAverageOrderValue
0,18102,580987.04,2578.40,578408.64,145,181645,712.0,180933.0,0.443796,4006.807172,3989.025103
1,14646,526751.52,3548.78,523202.74,145,367072,2116.0,364956.0,0.673710,3632.769103,3608.294759
7,16446,168472.50,168469.60,2.90,2,80997,80995.0,2.0,99.998279,84236.250000,1.450000
18,12346,77352.96,77183.60,169.36,3,74239,74215.0,24.0,99.781056,25784.320000,56.453333


In [147]:
top_customers_net_revenue = (
    customer_performance
    .sort_values(
        "NetRevenue",
        ascending=False
    )
    .head(20)
)

top_customers_net_revenue[
    [
        "Customer ID",
        "GrossSales",
        "ReturnValue",
        "NetRevenue",
        "Orders",
        "NetUnits",
        "NetAverageOrderValue",
        "RevenueReturnRatePct"
    ]
]

,Customer ID,GrossSales,ReturnValue,NetRevenue,Orders,NetUnits,NetAverageOrderValue,RevenueReturnRatePct
0,18102,580987.04,2578.40,578408.64,145,180933.0,3989.025103,0.443796
1,14646,526751.52,3548.78,523202.74,145,364956.0,3608.294759,0.673710
2,14156,303069.88,5294.87,297775.01,144,162209.0,2067.882014,1.747079
3,14911,272252.79,13730.36,258522.43,373,141354.0,693.089625,5.043239
4,17450,244784.25,11140.34,233643.91,51,80202.0,4581.253137,4.551085
5,13694,195640.69,5249.19,190391.50,143,184542.0,1331.409091,2.683077
6,17511,172132.87,3628.43,168504.44,60,115451.0,2808.407333,2.107924
9,12415,144033.37,926.35,143107.02,24,91016.0,5962.792500,0.643150
8,16684,147142.77,5612.48,141530.29,55,101096.0,2573.278000,3.814309
10,15061,126387.57,1405.44,124982.13,127,73601.0,984.111260,1.112008


In [148]:
total_known_customer_net_revenue = (
    customer_performance["NetRevenue"].sum()
)

customer_performance = (
    customer_performance
    .sort_values(
        "NetRevenue",
        ascending=False
    )
    .reset_index(drop=True)
)

customer_performance["NetRevenueSharePct"] = (
    customer_performance["NetRevenue"]
    / total_known_customer_net_revenue
    * 100
)

customer_performance["CumulativeNetRevenueSharePct"] = (
    customer_performance["NetRevenueSharePct"].cumsum()
)

In [149]:
customer_performance[
    [
        "Customer ID",
        "NetRevenue",
        "Orders",
        "NetUnits",
        "NetAverageOrderValue",
        "RevenueReturnRatePct",
        "NetRevenueSharePct",
        "CumulativeNetRevenueSharePct"
    ]
].head(20)

,Customer ID,NetRevenue,Orders,NetUnits,NetAverageOrderValue,RevenueReturnRatePct,NetRevenueSharePct,CumulativeNetRevenueSharePct
0,18102,578408.64,145,180933.0,3989.025103,0.443796,3.535505,3.535505
1,14646,523202.74,145,364956.0,3608.294759,0.673710,3.198061,6.733566
2,14156,297775.01,144,162209.0,2067.882014,1.747079,1.820141,8.553707
3,14911,258522.43,373,141354.0,693.089625,5.043239,1.580211,10.133918
4,17450,233643.91,51,80202.0,4581.253137,4.551085,1.428141,11.562059
5,13694,190391.50,143,184542.0,1331.409091,2.683077,1.163762,12.725821
6,17511,168504.44,60,115451.0,2808.407333,2.107924,1.029978,13.755800
7,12415,143107.02,24,91016.0,5962.792500,0.643150,0.874737,14.630537
8,16684,141530.29,55,101096.0,2573.278000,3.814309,0.865100,15.495637
9,15061,124982.13,127,73601.0,984.111260,1.112008,0.763950,16.259586


### Repeat Customer Analysis

Customers are classified as one-time or repeat purchasers based on the number of distinct merchandise orders placed.

Repeat-customer behavior is evaluated using customer counts and realized net revenue to understand the importance of recurring purchasing activity to the business.

In [150]:
customer_performance["CustomerType"] = (
    customer_performance["Orders"]
    .apply(
        lambda x: "Repeat Customer"
        if x > 1
        else "One-Time Customer"
    )
)

In [151]:
customer_type_summary = (
    customer_performance
    .groupby("CustomerType")
    .agg(
        Customers=("Customer ID", "nunique"),
        NetRevenue=("NetRevenue", "sum"),
        Orders=("Orders", "sum"),
        NetUnits=("NetUnits", "sum")
    )
    .reset_index()
)

customer_type_summary

,CustomerType,Customers,NetRevenue,Orders,NetUnits
0,One-Time Customer,1618,5.426278e+05,1618,421768.0
1,Repeat Customer,4234,1.581737e+07,34976,9613263.0


In [152]:
customer_type_summary["CustomerSharePct"] = (
    customer_type_summary["Customers"]
    / customer_type_summary["Customers"].sum()
    * 100
)

customer_type_summary["NetRevenueSharePct"] = (
    customer_type_summary["NetRevenue"]
    / customer_type_summary["NetRevenue"].sum()
    * 100
)

customer_type_summary

,CustomerType,Customers,NetRevenue,Orders,NetUnits,CustomerSharePct,NetRevenueSharePct
0,One-Time Customer,1618,5.426278e+05,1618,421768.0,27.648667,3.316796
1,Repeat Customer,4234,1.581737e+07,34976,9613263.0,72.351333,96.683204


In [153]:
repeat_customer_rate = (
    (
        customer_performance["Orders"] > 1
    ).mean()
    * 100
)

print(
    "Repeat Customer Rate:",
    round(repeat_customer_rate, 2),
    "%"
)

Repeat Customer Rate: 72.35 %


In [154]:
customer_value_by_type = (
    customer_performance
    .groupby("CustomerType")
    .agg(
        AvgNetRevenuePerCustomer=("NetRevenue", "mean"),
        MedianNetRevenuePerCustomer=("NetRevenue", "median"),
        AvgOrdersPerCustomer=("Orders", "mean"),
        AvgNetUnitsPerCustomer=("NetUnits", "mean")
    )
)

customer_value_by_type

,AvgNetRevenuePerCustomer,MedianNetRevenuePerCustomer,AvgOrdersPerCustomer,AvgNetUnitsPerCustomer
CustomerType,,,,
One-Time Customer,335.369470,223.95,1.000000,260.672435
Repeat Customer,3735.798544,1330.26,8.260746,2270.491970


### Repeat Customer Findings

Repeat purchasing is a major driver of customer value.

Approximately 72.35% of identified customers placed more than one merchandise order, while 27.65% purchased only once. Repeat customers generated approximately 96.68% of identified-customer net revenue, compared with only 3.32% from one-time customers.

Repeat customers also demonstrated substantially higher customer value, with average net revenue of approximately £3,735.80 per customer and an average of 8.26 orders, compared with approximately £335.37 and one order among one-time customers.

These findings indicate that recurring customer relationships represent a significant component of realized merchandise revenue.

### RFM Customer Segmentation

Identified customers are segmented using Recency, Frequency, and Monetary value to distinguish highly valuable and recently active customers from lower-frequency or inactive customer groups.

Recency is measured relative to the end of the available transaction period, Frequency represents distinct merchandise orders, and Monetary value represents realized net merchandise revenue after returns.

In [155]:
rfm_reference_date = (
    customer_sales["InvoiceDate"].max()
    + pd.Timedelta(days=1)
)

print("RFM Reference Date:", rfm_reference_date)

RFM Reference Date: 2011-12-10 12:50:00


In [156]:
rfm = customer_performance[
    [
        "Customer ID",
        "LastPurchase",
        "Orders",
        "NetRevenue"
    ]
].copy()

rfm["Recency"] = (
    rfm_reference_date
    - rfm["LastPurchase"]
).dt.days

rfm = rfm.rename(
    columns={
        "Orders": "Frequency",
        "NetRevenue": "Monetary"
    }
)

rfm.head()

,Customer ID,LastPurchase,Frequency,Monetary,Recency
0,18102,2011-12-09 11:50:00,145,578408.64,1
1,14646,2011-12-08 12:12:00,145,523202.74,2
2,14156,2011-11-30 10:54:00,144,297775.01,10
3,14911,2011-12-08 15:54:00,373,258522.43,1
4,17450,2011-12-01 13:29:00,51,233643.91,8


In [157]:
rfm[
    ["Recency", "Frequency", "Monetary"]
].describe()

,Recency,Frequency,Monetary
count,5852.000000,5852.000000,5852.000000
mean,200.198052,6.253247,2795.625228
std,208.509570,12.749286,13825.262580
min,1.000000,1.000000,-1343.240000
25%,25.000000,1.000000,330.942500
50%,95.000000,3.000000,840.245000
75%,379.000000,7.000000,2172.030000
max,739.000000,373.000000,578408.640000


In [158]:
print("Customers:", len(rfm))
print("Negative monetary customers:", (rfm["Monetary"] < 0).sum())
print("Zero monetary customers:", (rfm["Monetary"] == 0).sum())

print(
    "Recency range:",
    rfm["Recency"].min(),
    "to",
    rfm["Recency"].max()
)

Customers: 5852
Negative monetary customers: 3
Zero monetary customers: 16
Recency range: 1 to 739


In [159]:
rfm[["Recency", "Frequency", "Monetary"]].describe()

,Recency,Frequency,Monetary
count,5852.000000,5852.000000,5852.000000
mean,200.198052,6.253247,2795.625228
std,208.509570,12.749286,13825.262580
min,1.000000,1.000000,-1343.240000
25%,25.000000,1.000000,330.942500
50%,95.000000,3.000000,840.245000
75%,379.000000,7.000000,2172.030000
max,739.000000,373.000000,578408.640000


### RFM Scoring Treatment

RFM segmentation is performed on customers with positive realized net revenue.

Customers with zero or negative net revenue are retained in the analytical dataset but excluded from standard RFM scoring because their realized monetary value reflects fully reversed, returned, or otherwise non-positive purchasing activity.

RFM scores are based on relative customer distributions rather than arbitrary monetary thresholds.

In [160]:
rfm_positive = rfm[
    rfm["Monetary"] > 0
].copy()

rfm_nonpositive = rfm[
    rfm["Monetary"] <= 0
].copy()

print("RFM customers scored:", len(rfm_positive))
print("Non-positive customers excluded from RFM scoring:", len(rfm_nonpositive))

RFM customers scored: 5833
Non-positive customers excluded from RFM scoring: 19


In [161]:
rfm_positive["R_Score"] = pd.qcut(
    rfm_positive["Recency"].rank(method="first"),
    4,
    labels=[4, 3, 2, 1]
).astype(int)

rfm_positive["F_Score"] = pd.qcut(
    rfm_positive["Frequency"].rank(method="first"),
    4,
    labels=[1, 2, 3, 4]
).astype(int)

rfm_positive["M_Score"] = pd.qcut(
    rfm_positive["Monetary"].rank(method="first"),
    4,
    labels=[1, 2, 3, 4]
).astype(int)

In [162]:
rfm_positive["RFM_Total"] = (
    rfm_positive["R_Score"]
    + rfm_positive["F_Score"]
    + rfm_positive["M_Score"]
)

rfm_positive["RFM_Code"] = (
    rfm_positive["R_Score"].astype(str)
    + rfm_positive["F_Score"].astype(str)
    + rfm_positive["M_Score"].astype(str)
)

In [163]:
rfm_positive[
    [
        "Customer ID",
        "Recency",
        "Frequency",
        "Monetary",
        "R_Score",
        "F_Score",
        "M_Score",
        "RFM_Total",
        "RFM_Code"
    ]
].head(20)

,Customer ID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Total,RFM_Code
0,18102,1,145,578408.64,4,4,4,12,444
1,14646,2,145,523202.74,4,4,4,12,444
2,14156,10,144,297775.01,4,4,4,12,444
3,14911,1,373,258522.43,4,4,4,12,444
4,17450,8,51,233643.91,4,4,4,12,444
5,13694,4,143,190391.50,4,4,4,12,444
6,17511,3,60,168504.44,4,4,4,12,444
7,12415,24,24,143107.02,4,4,4,12,444
8,16684,4,55,141530.29,4,4,4,12,444
9,15061,4,127,124982.13,4,4,4,12,444


In [164]:
print(rfm_positive["R_Score"].value_counts().sort_index())
print()
print(rfm_positive["F_Score"].value_counts().sort_index())
print()
print(rfm_positive["M_Score"].value_counts().sort_index())

R_Score
1    1458
2    1458
3    1458
4    1459
Name: count, dtype: int64

F_Score
1    1459
2    1458
3    1458
4    1458
Name: count, dtype: int64

M_Score
1    1459
2    1458
3    1458
4    1458
Name: count, dtype: int64


### RFM Score Calibration

RFM scores are assigned using observed quartile thresholds from the customer distribution. This approach ensures that customers with identical behavioral values receive the same score rather than being separated only to create equally sized groups.

In [165]:
r_q1, r_q2, r_q3 = rfm_positive["Recency"].quantile([0.25, 0.50, 0.75])
f_q1, f_q2, f_q3 = rfm_positive["Frequency"].quantile([0.25, 0.50, 0.75])
m_q1, m_q2, m_q3 = rfm_positive["Monetary"].quantile([0.25, 0.50, 0.75])

print("Recency thresholds:", r_q1, r_q2, r_q3)
print("Frequency thresholds:", f_q1, f_q2, f_q3)
print("Monetary thresholds:", round(m_q1, 2), round(m_q2, 2), round(m_q3, 2))

Recency thresholds: 25.0 95.0 379.0
Frequency thresholds: 1.0 3.0 7.0
Monetary thresholds: 333.74 843.74 2179.76


In [166]:
def score_recency(x):
    if x <= r_q1:
        return 4
    elif x <= r_q2:
        return 3
    elif x <= r_q3:
        return 2
    else:
        return 1


def score_positive_metric(x, q1, q2, q3):
    if x <= q1:
        return 1
    elif x <= q2:
        return 2
    elif x <= q3:
        return 3
    else:
        return 4


rfm_positive["R_Score"] = rfm_positive["Recency"].apply(score_recency)

rfm_positive["F_Score"] = rfm_positive["Frequency"].apply(
    lambda x: score_positive_metric(x, f_q1, f_q2, f_q3)
)

rfm_positive["M_Score"] = rfm_positive["Monetary"].apply(
    lambda x: score_positive_metric(x, m_q1, m_q2, m_q3)
)

In [167]:
rfm_positive["RFM_Total"] = (
    rfm_positive["R_Score"]
    + rfm_positive["F_Score"]
    + rfm_positive["M_Score"]
)

rfm_positive["RFM_Code"] = (
    rfm_positive["R_Score"].astype(str)
    + rfm_positive["F_Score"].astype(str)
    + rfm_positive["M_Score"].astype(str)
)

In [168]:
import numpy as np

In [169]:
conditions = [
    # Best customers: recent, frequent and high value
    (
        (rfm_positive["R_Score"] == 4)
        & (rfm_positive["F_Score"] >= 3)
        & (rfm_positive["M_Score"] >= 3)
    ),

    # Strong recurring customers
    (
        (rfm_positive["R_Score"] >= 3)
        & (rfm_positive["F_Score"] >= 3)
        & (rfm_positive["M_Score"] >= 2)
    ),

    # Very recent but still early in relationship
    (
        (rfm_positive["R_Score"] == 4)
        & (rfm_positive["F_Score"] <= 2)
    ),

    # Relatively recent with growth potential
    (
        (rfm_positive["R_Score"] == 3)
        & (rfm_positive["F_Score"] <= 2)
        & (rfm_positive["M_Score"] >= 2)
    ),

    # Previously valuable but haven't purchased recently
    (
        (rfm_positive["R_Score"] <= 2)
        & (rfm_positive["M_Score"] >= 3)
    ),

    # Older customers with some historical repeat activity
    (
        (rfm_positive["R_Score"] <= 2)
        & (rfm_positive["F_Score"] >= 2)
    )
]

segment_names = [
    "Champions",
    "Loyal Customers",
    "Recent Customers",
    "Promising",
    "High-Value At Risk",
    "At Risk"
]

rfm_positive["CustomerSegment"] = np.select(
    conditions,
    segment_names,
    default="Low Engagement"
)

In [170]:
rfm_positive["CustomerSegment"].value_counts()

CustomerSegment
Low Engagement        1393
Champions             1005
High-Value At Risk     915
Loyal Customers        871
At Risk                826
Promising              436
Recent Customers       387
Name: count, dtype: int64

In [171]:
segment_summary = (
    rfm_positive
    .groupby("CustomerSegment")
    .agg(
        Customers=("Customer ID", "nunique"),
        AvgRecencyDays=("Recency", "mean"),
        AvgFrequency=("Frequency", "mean"),
        NetRevenue=("Monetary", "sum"),
        AvgCustomerValue=("Monetary", "mean")
    )
    .reset_index()
)

segment_summary["CustomerSharePct"] = (
    segment_summary["Customers"]
    / segment_summary["Customers"].sum()
    * 100
)

segment_summary["RevenueSharePct"] = (
    segment_summary["NetRevenue"]
    / segment_summary["NetRevenue"].sum()
    * 100
)

segment_summary = segment_summary.sort_values(
    "NetRevenue",
    ascending=False
)

segment_summary

,CustomerSegment,Customers,AvgRecencyDays,AvgFrequency,NetRevenue,AvgCustomerValue,CustomerSharePct,RevenueSharePct
1,Champions,1005,10.880597,17.951244,9633076.325,9585.150572,17.229556,58.876313
4,Loyal Customers,871,49.864524,8.817451,2970820.663,3410.815916,14.932282,18.157332
2,High-Value At Risk,915,290.691803,5.817486,2360677.457,2579.975363,15.686611,14.428203
5,Promising,436,54.972477,2.146789,406593.341,932.553534,7.474713,2.485054
0,At Risk,826,342.361985,2.762712,395434.670,478.734467,14.160809,2.416854
3,Low Engagement,1393,383.070352,1.078966,330782.621,237.460604,23.881365,2.021707
6,Recent Customers,387,13.385013,2.090439,264163.220,682.592300,6.634665,1.614537


### RFM Segmentation Findings

Customer value is highly concentrated among recent and frequently purchasing customers.

- Champions represent approximately 17.23% of scored customers but generate approximately 58.88% of identified-customer net revenue.
- Champions and Loyal Customers together represent approximately 32.16% of customers while contributing approximately 77.03% of net customer revenue.
- High-Value At Risk customers account for approximately 15.69% of customers and 14.43% of historical net customer revenue, with an average recency of approximately 291 days.
- Low Engagement customers represent the largest individual segment at approximately 23.88% of customers but contribute only approximately 2.02% of net customer revenue.
- Recent and Promising customers represent potential development opportunities but currently contribute a relatively small proportion of total customer revenue.

These results suggest differentiated customer strategies: protect high-value active customers, re-engage historically valuable inactive customers, nurture promising customers, and avoid disproportionately allocating retention resources to low-engagement customers.

## Geographic Performance Analysis

Merchandise sales are evaluated across countries to understand geographic revenue concentration, transaction activity, unit demand, and average order value.

Country-level analysis supports identification of core markets, international revenue opportunities, and geographic concentration risk. These metrics will later support an interactive geographic dashboard.

In [172]:
country_performance = (
    merchandise_sales
    .groupby("Country")
    .agg(
        GrossSales=("GrossSales", "sum"),
        Orders=("Invoice", "nunique"),
        UnitsSold=("UnitsSold", "sum"),
        Customers=("Customer ID", "nunique")
    )
    .reset_index()
)

In [173]:
country_performance["AverageOrderValue"] = (
    country_performance["GrossSales"]
    / country_performance["Orders"]
)

In [174]:
country_performance["RevenueSharePct"] = (
    country_performance["GrossSales"]
    / country_performance["GrossSales"].sum()
    * 100
)

In [175]:
country_performance = country_performance.sort_values(
    "GrossSales",
    ascending=False
).reset_index(drop=True)

country_performance.head(20)

,Country,GrossSales,Orders,UnitsSold,Customers,AverageOrderValue,RevenueSharePct
0,United Kingdom,1.680278e+07,36184,9176270,5334,464.370317,85.529382
1,EIRE,6.234142e+05,581,336088,3,1073.001997,3.173299
2,Netherlands,5.497734e+05,216,383625,22,2545.247269,2.798453
3,Germany,3.832890e+05,753,223089,107,509.015938,1.951015
4,France,3.110903e+05,598,270582,93,520.217876,1.583510
5,Australia,1.678000e+05,89,103753,15,1885.393371,0.854135
6,Spain,9.776675e+04,144,49996,38,678.935764,0.497652
7,Switzerland,9.402459e+04,85,52612,22,1106.171647,0.478603
8,Sweden,8.631914e+04,99,88537,19,871.910505,0.439381
9,Denmark,6.742269e+04,42,237406,12,1605.302143,0.343195


In [176]:
uk_sales = country_performance.loc[
    country_performance["Country"] == "United Kingdom",
    "GrossSales"
].iloc[0]

total_sales = country_performance["GrossSales"].sum()

uk_revenue_share = (
    uk_sales / total_sales * 100
)

print(
    "United Kingdom Revenue Share:",
    round(uk_revenue_share, 2),
    "%"
)

United Kingdom Revenue Share: 85.53 %


In [177]:
international_sales = (
    total_sales - uk_sales
)

international_revenue_share = (
    international_sales
    / total_sales
    * 100
)

print(
    "International Revenue:",
    round(international_sales, 2)
)

print(
    "International Revenue Share:",
    round(international_revenue_share, 2),
    "%"
)

International Revenue: 2842842.25
International Revenue Share: 14.47 %


In [178]:
international_markets = (
    country_performance[
        country_performance["Country"] != "United Kingdom"
    ]
    .sort_values(
        "GrossSales",
        ascending=False
    )
)

international_markets[
    [
        "Country",
        "GrossSales",
        "RevenueSharePct",
        "Orders",
        "UnitsSold",
        "Customers",
        "AverageOrderValue"
    ]
].head(15)

,Country,GrossSales,RevenueSharePct,Orders,UnitsSold,Customers,AverageOrderValue
1,EIRE,623414.160,3.173299,581,336088,3,1073.001997
2,Netherlands,549773.410,2.798453,216,383625,22,2545.247269
3,Germany,383289.001,1.951015,753,223089,107,509.015938
4,France,311090.290,1.583510,598,270582,93,520.217876
5,Australia,167800.010,0.854135,89,103753,15,1885.393371
6,Spain,97766.750,0.497652,144,49996,38,678.935764
7,Switzerland,94024.590,0.478603,85,52612,22,1106.171647
8,Sweden,86319.140,0.439381,99,88537,19,871.910505
9,Denmark,67422.690,0.343195,42,237406,12,1605.302143
10,Belgium,56993.170,0.290106,143,34347,29,398.553636


In [179]:
sorted(
    merchandise_sales["Country"]
    .dropna()
    .unique()
)

['Australia',
 'Austria',
 'Bahrain',
 'Belgium',
 'Bermuda',
 'Brazil',
 'Canada',
 'Channel Islands',
 'Cyprus',
 'Czech Republic',
 'Denmark',
 'EIRE',
 'European Community',
 'Finland',
 'France',
 'Germany',
 'Greece',
 'Hong Kong',
 'Iceland',
 'Israel',
 'Italy',
 'Japan',
 'Korea',
 'Lebanon',
 'Lithuania',
 'Malta',
 'Netherlands',
 'Nigeria',
 'Norway',
 'Poland',
 'Portugal',
 'RSA',
 'Saudi Arabia',
 'Singapore',
 'Spain',
 'Sweden',
 'Switzerland',
 'Thailand',
 'USA',
 'United Arab Emirates',
 'United Kingdom',
 'Unspecified',
 'West Indies']

### Geographic Standardization

Country labels are standardized for geographic analysis and dashboard mapping while preserving the original source values.

Clearly identifiable legacy abbreviations are converted to modern country names. Regional, unspecified, or ambiguous geographic labels are retained separately and flagged so they do not distort country-level map analysis.

In [180]:
country_mapping = {
    "EIRE": "Ireland",
    "USA": "United States",
    "RSA": "South Africa"
}

sales_working["CountryStandardized"] = (
    sales_working["Country"]
    .replace(country_mapping)
)

In [181]:
non_country_labels = [
    "European Community",
    "Unspecified",
    "West Indies"
]

sales_working["GeographicStatus"] = "Mapped Country"

sales_working.loc[
    sales_working["CountryStandardized"].isin(non_country_labels),
    "GeographicStatus"
] = "Non-Country / Unspecified"

sales_working.loc[
    sales_working["CountryStandardized"] == "Korea",
    "GeographicStatus"
] = "Requires Geographic Review"

In [182]:
sales_working[
    [
        "Country",
        "CountryStandardized",
        "GeographicStatus"
    ]
].drop_duplicates().sort_values(
    "CountryStandardized"
)

,Country,CountryStandardized,GeographicStatus
178,Australia,Australia,Mapped Country
17287,Austria,Austria,Mapped Country
79102,Bahrain,Bahrain,Mapped Country
173,Belgium,Belgium,Mapped Country
128120,Bermuda,Bermuda,Mapped Country
348243,Brazil,Brazil,Mapped Country
416551,Canada,Canada,Mapped Country
9628,Channel Islands,Channel Islands,Mapped Country
12263,Cyprus,Cyprus,Mapped Country
606536,Czech Republic,Czech Republic,Mapped Country


In [183]:
merchandise_sales["CountryStandardized"] = (
    merchandise_sales["Country"]
    .replace(country_mapping)
)

merchandise_returns["CountryStandardized"] = (
    merchandise_returns["Country"]
    .replace(country_mapping)
)

In [184]:
country_sales = (
    merchandise_sales
    .groupby("CountryStandardized")
    .agg(
        GrossSales=("GrossSales", "sum"),
        Orders=("Invoice", "nunique"),
        UnitsSold=("UnitsSold", "sum"),
        Customers=("Customer ID", "nunique")
    )
    .reset_index()
)

In [185]:
country_returns = (
    merchandise_returns
    .groupby("CountryStandardized")
    .agg(
        ReturnValue=("ReturnValue", "sum"),
        ReturnedUnits=("ReturnedUnits", "sum"),
        ReturnInvoices=("Invoice", "nunique")
    )
    .reset_index()
)

In [186]:
country_performance = country_sales.merge(
    country_returns,
    on="CountryStandardized",
    how="left"
)

country_performance[
    ["ReturnValue", "ReturnedUnits", "ReturnInvoices"]
] = country_performance[
    ["ReturnValue", "ReturnedUnits", "ReturnInvoices"]
].fillna(0)

In [187]:
country_performance["NetRevenue"] = (
    country_performance["GrossSales"]
    - country_performance["ReturnValue"]
)

country_performance["NetUnits"] = (
    country_performance["UnitsSold"]
    - country_performance["ReturnedUnits"]
)

country_performance["AverageOrderValue"] = (
    country_performance["NetRevenue"]
    / country_performance["Orders"]
)

country_performance["RevenueReturnRatePct"] = (
    country_performance["ReturnValue"]
    / country_performance["GrossSales"]
    * 100
)

In [188]:
map_country_performance = country_performance[
    ~country_performance["CountryStandardized"].isin(
        non_country_labels
    )
].copy()

In [189]:
map_country_performance = map_country_performance[
    map_country_performance["CountryStandardized"] != "Korea"
].copy()

In [190]:
map_country_performance["NetRevenueSharePct"] = (
    map_country_performance["NetRevenue"]
    / map_country_performance["NetRevenue"].sum()
    * 100
)

map_country_performance = (
    map_country_performance
    .sort_values(
        "NetRevenue",
        ascending=False
    )
    .reset_index(drop=True)
)

In [191]:
map_country_performance[
    [
        "CountryStandardized",
        "GrossSales",
        "ReturnValue",
        "NetRevenue",
        "RevenueReturnRatePct",
        "Orders",
        "NetUnits",
        "Customers",
        "AverageOrderValue",
        "NetRevenueSharePct"
    ]
].head(20)

,CountryStandardized,GrossSales,ReturnValue,NetRevenue,RevenueReturnRatePct,Orders,NetUnits,Customers,AverageOrderValue,NetRevenueSharePct
0,United Kingdom,1.680278e+07,632183.00,1.617059e+07,3.762372,36184,8830347.0,5334,446.898976,85.488384
1,Ireland,6.234142e+05,20297.55,6.031166e+05,3.255869,581,326954.0,3,1038.066454,3.188471
2,Netherlands,5.497734e+05,3677.98,5.460954e+05,0.668999,216,381461.0,22,2528.219583,2.887019
3,Germany,3.832890e+05,8715.49,3.745735e+05,2.273869,753,219786.0,107,497.441582,1.980242
4,France,3.110903e+05,17659.09,2.934312e+05,5.676516,598,180308.0,93,490.687625,1.551270
5,Australia,1.678000e+05,1517.86,1.662822e+05,0.904565,89,103064.0,15,1868.338764,0.879077
6,Switzerland,9.402459e+04,1106.53,9.291806e+04,1.176852,85,52113.0,22,1093.153647,0.491226
7,Spain,9.776675e+04,12955.19,8.481156e+04,13.251121,144,44356.0,38,588.969167,0.448370
8,Sweden,8.631914e+04,1973.97,8.434517e+04,2.286828,99,87773.0,19,851.971414,0.445904
9,Denmark,6.742269e+04,4067.10,6.335559e+04,6.032242,42,234702.0,12,1508.466429,0.334939


In [192]:
uk_net_revenue = map_country_performance.loc[
    map_country_performance["CountryStandardized"]
    == "United Kingdom",
    "NetRevenue"
].iloc[0]

total_mapped_net_revenue = (
    map_country_performance["NetRevenue"].sum()
)

uk_net_revenue_share = (
    uk_net_revenue
    / total_mapped_net_revenue
    * 100
)

international_net_revenue = (
    total_mapped_net_revenue - uk_net_revenue
)

international_net_revenue_share = (
    international_net_revenue
    / total_mapped_net_revenue
    * 100
)

print(
    "UK Net Revenue Share:",
    round(uk_net_revenue_share, 2),
    "%"
)

print(
    "International Net Revenue Share:",
    round(international_net_revenue_share, 2),
    "%"
)

UK Net Revenue Share: 85.49 %
International Net Revenue Share: 14.51 %


### International Market Performance

Because the United Kingdom represents the majority of realized merchandise revenue, international markets are evaluated separately to identify meaningful secondary markets, differences in average order value, and potential geographic expansion opportunities.

In [193]:
international_performance = (
    map_country_performance[
        map_country_performance["CountryStandardized"]
        != "United Kingdom"
    ]
    .copy()
)

international_performance[
    [
        "CountryStandardized",
        "NetRevenue",
        "NetRevenueSharePct",
        "Orders",
        "NetUnits",
        "Customers",
        "AverageOrderValue",
        "RevenueReturnRatePct"
    ]
].head(15)

,CountryStandardized,NetRevenue,NetRevenueSharePct,Orders,NetUnits,Customers,AverageOrderValue,RevenueReturnRatePct
1,Ireland,603116.610,3.188471,581,326954.0,3,1038.066454,3.255869
2,Netherlands,546095.430,2.887019,216,381461.0,22,2528.219583,0.668999
3,Germany,374573.511,1.980242,753,219786.0,107,497.441582,2.273869
4,France,293431.200,1.551270,598,180308.0,93,490.687625,5.676516
5,Australia,166282.150,0.879077,89,103064.0,15,1868.338764,0.904565
6,Switzerland,92918.060,0.491226,85,52113.0,22,1093.153647,1.176852
7,Spain,84811.560,0.448370,144,44356.0,38,588.969167,13.251121
8,Sweden,84345.170,0.445904,99,87773.0,19,851.971414,2.286828
9,Denmark,63355.590,0.334939,42,234702.0,12,1508.466429,6.032242
10,Belgium,56598.890,0.299219,143,34188.0,29,395.796434,0.691802


In [194]:
international_performance["InternationalRevenueSharePct"] = (
    international_performance["NetRevenue"]
    / international_performance["NetRevenue"].sum()
    * 100
)

international_performance[
    [
        "CountryStandardized",
        "NetRevenue",
        "InternationalRevenueSharePct",
        "Orders",
        "Customers",
        "AverageOrderValue",
        "RevenueReturnRatePct"
    ]
].head(15)

,CountryStandardized,NetRevenue,InternationalRevenueSharePct,Orders,Customers,AverageOrderValue,RevenueReturnRatePct
1,Ireland,603116.610,21.971854,581,3,1038.066454,3.255869
2,Netherlands,546095.430,19.894542,216,22,2528.219583,0.668999
3,Germany,374573.511,13.645909,753,107,497.441582,2.273869
4,France,293431.200,10.689852,598,93,490.687625,5.676516
5,Australia,166282.150,6.057746,89,15,1868.338764,0.904565
6,Switzerland,92918.060,3.385054,85,22,1093.153647,1.176852
7,Spain,84811.560,3.089729,144,38,588.969167,13.251121
8,Sweden,84345.170,3.072739,99,19,851.971414,2.286828
9,Denmark,63355.590,2.308077,42,12,1508.466429,6.032242
10,Belgium,56598.890,2.061927,143,29,395.796434,0.691802


In [195]:
international_performance["InternationalRevenueSharePct"] = (
    international_performance["NetRevenue"]
    / international_performance["NetRevenue"].sum()
    * 100
)

international_performance[
    [
        "CountryStandardized",
        "NetRevenue",
        "InternationalRevenueSharePct",
        "Orders",
        "Customers",
        "AverageOrderValue",
        "RevenueReturnRatePct"
    ]
].head(15)

,CountryStandardized,NetRevenue,InternationalRevenueSharePct,Orders,Customers,AverageOrderValue,RevenueReturnRatePct
1,Ireland,603116.610,21.971854,581,3,1038.066454,3.255869
2,Netherlands,546095.430,19.894542,216,22,2528.219583,0.668999
3,Germany,374573.511,13.645909,753,107,497.441582,2.273869
4,France,293431.200,10.689852,598,93,490.687625,5.676516
5,Australia,166282.150,6.057746,89,15,1868.338764,0.904565
6,Switzerland,92918.060,3.385054,85,22,1093.153647,1.176852
7,Spain,84811.560,3.089729,144,38,588.969167,13.251121
8,Sweden,84345.170,3.072739,99,19,851.971414,2.286828
9,Denmark,63355.590,2.308077,42,12,1508.466429,6.032242
10,Belgium,56598.890,2.061927,143,29,395.796434,0.691802


In [196]:
international_performance[
    [
        "CountryStandardized",
        "NetRevenue",
        "Orders",
        "Customers",
        "AverageOrderValue",
        "RevenueReturnRatePct"
    ]
].sort_values(
    "AverageOrderValue",
    ascending=False
).head(15)

,CountryStandardized,NetRevenue,Orders,Customers,AverageOrderValue,RevenueReturnRatePct
2,Netherlands,546095.43,216,22,2528.219583,0.668999
5,Australia,166282.15,89,15,1868.338764,0.904565
21,Singapore,13158.16,8,1,1644.770000,0.000000
31,Thailand,3070.54,2,1,1535.270000,0.000000
20,Hong Kong,13655.50,9,0,1517.277778,0.080123
9,Denmark,63355.59,42,12,1508.466429,6.032242
35,Bermuda,1253.14,1,0,1253.140000,0.000000
13,Japan,39746.77,33,10,1204.447576,7.617020
30,South Africa,3369.03,3,2,1123.010000,0.000000
22,Israel,11101.37,10,4,1110.137000,2.007625


In [197]:
established_international_markets = (
    international_performance[
        international_performance["Orders"] >= 20
    ]
    .sort_values(
        "AverageOrderValue",
        ascending=False
    )
)

established_international_markets[
    [
        "CountryStandardized",
        "NetRevenue",
        "Orders",
        "Customers",
        "AverageOrderValue",
        "RevenueReturnRatePct"
    ]
].head(15)

,CountryStandardized,NetRevenue,Orders,Customers,AverageOrderValue,RevenueReturnRatePct
2,Netherlands,546095.430,216,22,2528.219583,0.668999
5,Australia,166282.150,89,15,1868.338764,0.904565
9,Denmark,63355.590,42,12,1508.466429,6.032242
13,Japan,39746.770,33,10,1204.447576,7.617020
6,Switzerland,92918.060,85,22,1093.153647,1.176852
1,Ireland,603116.610,581,3,1038.066454,3.255869
14,Norway,38413.680,39,12,984.966154,0.419128
8,Sweden,84345.170,99,19,851.971414,2.286828
12,Channel Islands,41841.810,54,13,774.848333,4.970483
17,Cyprus,23987.560,35,11,685.358857,2.028650


In [198]:
international_performance[
    international_performance["GrossSales"] >= 10000
][
    [
        "CountryStandardized",
        "GrossSales",
        "ReturnValue",
        "NetRevenue",
        "RevenueReturnRatePct",
        "Orders"
    ]
].sort_values(
    "RevenueReturnRatePct",
    ascending=False
).head(15)

,CountryStandardized,GrossSales,ReturnValue,NetRevenue,RevenueReturnRatePct,Orders
7,Spain,97766.750,12955.19,84811.560,13.251121,144
13,Japan,43023.910,3277.14,39746.770,7.617020,33
24,United Arab Emirates,10273.230,645.08,9628.150,6.279233,12
9,Denmark,67422.690,4067.10,63355.590,6.032242,42
4,France,311090.290,17659.09,293431.200,5.676516,598
12,Channel Islands,44030.330,2188.52,41841.810,4.970483,54
23,Poland,10214.290,374.36,9839.930,3.665061,28
1,Ireland,623414.160,20297.55,603116.610,3.255869,581
15,Italy,28937.500,710.70,28226.800,2.455983,59
8,Sweden,86319.140,1973.97,84345.170,2.286828,99


### Geographic Concentration Finding

The United Kingdom generates approximately 85.49% of mapped net merchandise revenue, while international markets collectively contribute approximately 14.51%.

This indicates substantial geographic revenue concentration in the domestic UK market. International markets provide additional revenue diversification, but the business remains highly dependent on UK demand. International market performance should therefore be evaluated separately to identify countries with sufficient revenue, order volume, and customer activity to support potential growth opportunities.

### International Return Risk

Return behavior varies meaningfully across international markets.

Among markets generating at least £10,000 in gross merchandise sales, Spain shows the highest revenue return rate at approximately 13.25%. Japan, the United Arab Emirates, Denmark, and France also exhibit comparatively elevated return rates.

Larger international markets such as Ireland and Germany generate substantially more revenue while maintaining lower return rates of approximately 3.26% and 2.27%, respectively.

These differences suggest that international market performance should be evaluated using both revenue growth and return behavior rather than sales volume alone.

In [199]:
international_performance[
    [
        "CountryStandardized",
        "NetRevenue",
        "InternationalRevenueSharePct",
        "Orders",
        "Customers",
        "AverageOrderValue",
        "RevenueReturnRatePct"
    ]
].sort_values(
    "NetRevenue",
    ascending=False
).head(15)

,CountryStandardized,NetRevenue,InternationalRevenueSharePct,Orders,Customers,AverageOrderValue,RevenueReturnRatePct
1,Ireland,603116.610,21.971854,581,3,1038.066454,3.255869
2,Netherlands,546095.430,19.894542,216,22,2528.219583,0.668999
3,Germany,374573.511,13.645909,753,107,497.441582,2.273869
4,France,293431.200,10.689852,598,93,490.687625,5.676516
5,Australia,166282.150,6.057746,89,15,1868.338764,0.904565
6,Switzerland,92918.060,3.385054,85,22,1093.153647,1.176852
7,Spain,84811.560,3.089729,144,38,588.969167,13.251121
8,Sweden,84345.170,3.072739,99,19,851.971414,2.286828
9,Denmark,63355.590,2.308077,42,12,1508.466429,6.032242
10,Belgium,56598.890,2.061927,143,29,395.796434,0.691802


In [200]:
established_international_markets[
    [
        "CountryStandardized",
        "NetRevenue",
        "Orders",
        "Customers",
        "AverageOrderValue",
        "RevenueReturnRatePct"
    ]
].head(15)

,CountryStandardized,NetRevenue,Orders,Customers,AverageOrderValue,RevenueReturnRatePct
2,Netherlands,546095.430,216,22,2528.219583,0.668999
5,Australia,166282.150,89,15,1868.338764,0.904565
9,Denmark,63355.590,42,12,1508.466429,6.032242
13,Japan,39746.770,33,10,1204.447576,7.617020
6,Switzerland,92918.060,85,22,1093.153647,1.176852
1,Ireland,603116.610,581,3,1038.066454,3.255869
14,Norway,38413.680,39,12,984.966154,0.419128
8,Sweden,84345.170,99,19,851.971414,2.286828
12,Channel Islands,41841.810,54,13,774.848333,4.970483
17,Cyprus,23987.560,35,11,685.358857,2.028650


### International Market Findings

International revenue is concentrated among a relatively small group of countries.

Ireland, the Netherlands, Germany, and France collectively contribute approximately 66% of international net merchandise revenue. Ireland and the Netherlands are the two largest international markets, together representing more than 40% of international revenue.

The Netherlands also demonstrates particularly strong order economics, with an average order value of approximately £2,528, followed by Australia and Denmark. This indicates that some smaller international markets generate substantial value through relatively large transactions.

Return behavior varies materially across countries. Spain records the highest return rate among established international markets at approximately 13.25%, while major markets such as the Netherlands, Australia, Germany, and Ireland maintain substantially lower rates.

These results suggest that international growth opportunities should be evaluated using a combination of net revenue, order value, customer activity, and return risk rather than revenue alone.

# Retail Sales Performance Analysis — Data Preparation

## Objective

Create a clean, documented, analysis-ready transaction dataset based on the data-quality and business-rule decisions established during exploratory profiling.

The processed dataset will serve as the common analytical source for SQL analysis, Excel reporting, and Tableau dashboard development.